# Van-Robot Collaborative Optimization — DRT Research Notebook

Implements a Mixed-Integer Linear Program (MILP) using AMPL + Gurobi to optimize
collaborative van–robot delivery routing across four experimental weeks:

1. **Week 1** — Deterministic baseline
2. **Week 2** — Deterministic simulation / stress testing
3. **Week 3** — Robust optimization
4. **Week 4** — Comparative analysis

---

## Setup

Install dependencies and initialize the AMPL + Gurobi solver.

In [ ]:
# Cell 1: Install (only needed once per session)
!pip install -q amplpy amplpy-gurobi --upgrade

In [ ]:
# Cell 2: Initialize NIU license + Gurobi
from amplpy import AMPL, ampl_notebook
import time
import matplotlib.pyplot as plt
import numpy as np

ampl = ampl_notebook(
    modules=["gurobi"],
    license_uuid="40a0a160-668d-492d-be4a-236079be8ff7"
)

print("✅ AMPL + Gurobi ready for ISYE 671!")
print("Gurobi version:", ampl.option['gurobi_version'])

## Shared Utilities

Single canonical definitions of all shared data structures and helper functions.
Every downstream section imports from this cell — do **not** redefine these elsewhere.

| Symbol | Purpose |
|--------|---------|
| `VanRobotData` | Dataclass holding all instance parameters |
| `build_sets` | Construct node/arc sets from data |
| `euclidean` | Euclidean distance between two nodes |
| `generate_random_euclidean_instance` | Random instance generator |
| `validate_data_keys` | Instance integrity check |
| `write_mod_file` / `write_dat` | Write AMPL `.mod` / `.dat` files |
| `solve_model_ampl` / `extract_solution` | Solve and parse AMPL results |

In [ ]:
#!/usr/bin/env python3
"""
AMPL VERSION of the compact van-robot MILP (Gurobi solver via amplpy) — FINAL FIXED VERSION

Matches your latest DOcplex code exactly (same formulation, same preprocessing,
same deterministic/nominal ρ logic, same Monte Carlo).

The only change needed was removing the duplicate "param bigM_n" from the .dat file
(it is defined once in the .mod file). Everything else is identical.

Requirements (run these two cells first in your notebook):
!pip install -q amplpy amplpy-gurobi --upgrade
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(modules=["gurobi"], license_uuid="40a0a160-668d-492d-be4a-236079be8ff7")
"""

from __future__ import annotations

import csv
import math
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple, Any

from amplpy import AMPL


# ============================================================
# Data container (unchanged — matches your latest version)
# ============================================================

@dataclass
class VanRobotData:
    """Sets and parameters exactly as in the original (and PDF)."""
    n: int

    cV: Dict[Tuple[int, int], float]
    tauV: Dict[Tuple[int, int], float]
    sV: Dict[int, float]

    tauR_out: Dict[Tuple[int, int], float]
    sR: Dict[int, float]
    tauR_in: Dict[Tuple[int, int], float]

    cR: Dict[Tuple[int, int, int], float]

    delta_out: Dict[Tuple[int, int], float]
    delta_srv: Dict[int, float]
    delta_in: Dict[Tuple[int, int], float]

    T: Dict[int, float]
    E_R: float
    H: float
    gamma: float
    lam: float

    coords: Optional[Dict[int, Tuple[float, float]]] = None
    alpha: Optional[float] = None


# ============================================================
# Set construction (unchanged)
# ============================================================

def build_sets(data: VanRobotData):
    start = 0
    end = data.n + 1
    C = list(range(1, data.n + 1))
    V = [start] + C + [end]

    A = [(k, l) for k in V for l in V if k != l and l != start and k != end]
    Fe = [(k, i, l) for (k, l) in A for i in C if i != k and i != l]

    return V, C, A, Fe, start, end


# ============================================================
# Geometry and random instance generation (unchanged — your latest version)
# ============================================================

def euclidean(p: Tuple[float, float], q: Tuple[float, float]) -> float:
    """Compute Euclidean distance between two points."""
    return math.hypot(p[0] - q[0], p[1] - q[1])


def generate_random_euclidean_instance(
    n: int,
    alpha: float,
    seed: int = 0,
    area_size: float = 100.0,
    van_speed: float = 1.0,
    robot_speed: float = 0.8,
    van_cost_per_distance: float = 1.0,
    robot_cost_per_distance: float = 0.6,
    customer_service_time_van: float = 2.0,
    customer_service_time_robot: float = 1.0,
    robot_endurance_factor: float = 1.8,
    gamma: float = 1.0,
    lam: float = 0.05,
    horizon_factor: float = 3.0,
) -> VanRobotData:
    """
    Generate a Euclidean random instance with coordinate-based travel times.

    Uncertainty construction using alpha in {0.05, 0.10, 0.20}:
    delta_out[(k,i)] = alpha * tauR_out[(k,i)]
    delta_in[(i,l)] = alpha * tauR_in[(i,l)]
    delta_srv[i] = alpha * sR[i]
    """
    allowed_alphas = {0.05, 0.10, 0.20}
    if alpha not in allowed_alphas:
        raise ValueError(f"alpha must be one of {sorted(allowed_alphas)}, got {alpha}")
    if van_speed <= 0 or robot_speed <= 0:
        raise ValueError("van_speed and robot_speed must be positive")

    rng = random.Random(seed)

    start = 0
    end = n + 1
    C = list(range(1, n + 1))
    V = [start] + C + [end]

    coords = {k: (rng.uniform(0.0, area_size), rng.uniform(0.0, area_size)) for k in V}
    A = [(k, l) for k in V for l in V if k != l and l != start and k != end]

    cV: Dict[Tuple[int, int], float] = {}
    tauV: Dict[Tuple[int, int], float] = {}
    for (k, l) in A:
        d = euclidean(coords[k], coords[l])
        cV[(k, l)] = van_cost_per_distance * d
        tauV[(k, l)] = d / van_speed

    sV = {k: customer_service_time_van for k in V}
    sV[start] = 0.0
    sV[end] = 0.0

    tauR_out: Dict[Tuple[int, int], float] = {}
    tauR_in: Dict[Tuple[int, int], float] = {}
    delta_out: Dict[Tuple[int, int], float] = {}
    delta_in: Dict[Tuple[int, int], float] = {}
    sR = {i: customer_service_time_robot for i in C}
    delta_srv = {i: alpha * sR[i] for i in C}

    for k in V:
        for i in C:
            if i != k:
                d = euclidean(coords[k], coords[i])
                tauR_out[(k, i)] = d / robot_speed
                delta_out[(k, i)] = alpha * tauR_out[(k, i)]

    for i in C:
        for l in V:
            if i != l:
                d = euclidean(coords[i], coords[l])
                tauR_in[(i, l)] = d / robot_speed
                delta_in[(i, l)] = alpha * tauR_in[(i, l)]

    cR: Dict[Tuple[int, int, int], float] = {}
    nominal_sortie_durations: List[float] = []

    for (k, l) in A:
        for i in C:
            if i != k and i != l:
                sortie_dist = euclidean(coords[k], coords[i]) + euclidean(coords[i], coords[l])
                cR[(k, i, l)] = robot_cost_per_distance * sortie_dist
                nominal_sortie_durations.append(
                    tauR_out[(k, i)] + sR[i] + tauR_in[(i, l)]
                )

    avg_nominal_sortie = sum(nominal_sortie_durations) / max(1, len(nominal_sortie_durations))
    E_R = robot_endurance_factor * avg_nominal_sortie

    avg_tauV = sum(tauV.values()) / max(1, len(tauV))
    H = horizon_factor * ((n + 1) * avg_tauV + sum(sV[k] for k in C) + E_R)
    if H <= 0:
        H = 1000.0
    T = {k: H for k in V}

    return VanRobotData(
        n=n,
        cV=cV,
        tauV=tauV,
        sV=sV,
        tauR_out=tauR_out,
        sR=sR,
        tauR_in=tauR_in,
        cR=cR,
        delta_out=delta_out,
        delta_srv=delta_srv,
        delta_in=delta_in,
        T=T,
        E_R=E_R,
        H=H,
        gamma=gamma,
        lam=lam,
        coords=coords,
        alpha=alpha,
    )


# ============================================================
# Validation, sortie builders & preprocessing (unchanged — your latest version)
# ============================================================

def validate_data_keys(data: VanRobotData, strict: bool = True) -> List[str]:
    """
    Validate that all required data keys are present.
    Debug check for missing keys in the data structure.
    """
    V, C, A, Fe, _, _ = build_sets(data)
    missing: List[str] = []

    for k in V:
        if k not in data.sV: missing.append(f"sV missing key [{k}]")
        if k not in data.T: missing.append(f"T missing key [{k}]")
    for i in C:
        if i not in data.sR: missing.append(f"sR missing key [{i}]")
        if i not in data.delta_srv: missing.append(f"delta_srv missing key [{i}]")
    for (k, l) in A:
        if (k, l) not in data.cV: missing.append(f"cV missing key [{k},{l}]")
        if (k, l) not in data.tauV: missing.append(f"tauV missing key [{k},{l}]")
    for (k, i, l) in Fe:
        if (k, i) not in data.tauR_out: missing.append(f"tauR_out missing key [{k},{i}]")
        if (i, l) not in data.tauR_in: missing.append(f"tauR_in missing key [{i},{l}]")
        if (k, i, l) not in data.cR: missing.append(f"cR missing key [{k},{i},{l}]")
        if (k, i) not in data.delta_out: missing.append(f"delta_out missing key [{k},{i}]")
        if (i, l) not in data.delta_in: missing.append(f"delta_in missing key [{i},{l}]")

    if missing and strict:
        preview = "\n".join(missing[:50])
        more = "" if len(missing) <= 50 else f"\n... and {len(missing) - 50} more missing keys"
        raise KeyError(f"Data validation failed with missing keys:\n{preview}{more}")
    return missing


def compute_nominal_sortie_duration(data: VanRobotData, Fe: Iterable[Tuple[int, int, int]]):
    rho_nom: Dict[Tuple[int, int, int], float] = {}
    for (k, i, l) in Fe:
        rho_nom[(k, i, l)] = data.tauR_out[(k, i)] + data.sR[i] + data.tauR_in[(i, l)]
    return rho_nom


def compute_robust_sortie_duration(data: VanRobotData, Fe: Iterable[Tuple[int, int, int]]):
    rho_rob: Dict[Tuple[int, int, int], float] = {}
    for (k, i, l) in Fe:
        rho_rob[(k, i, l)] = (
            data.tauR_out[(k, i)] + data.sR[i] + data.tauR_in[(i, l)]
            + data.delta_out[(k, i)] + data.delta_srv[i] + data.delta_in[(i, l)]
        )
    return rho_rob


def preprocess_sorties_from_duration(
    data: VanRobotData,
    A: Iterable[Tuple[int, int]],
    Fe: Iterable[Tuple[int, int, int]],
    duration: Dict[Tuple[int, int, int], float],
):
    """
    Automatic preprocessing of feasible sortie set F from a given sortie duration map.
    """
    F = [(k, i, l) for (k, i, l) in Fe if duration[(k, i, l)] <= data.E_R]

    W: Dict[Tuple[int, int], float] = {}
    for (k, l) in A:
        vals = [duration[(k, i, l)] for (kk, i, ll) in F if kk == k and ll == l]
        W[(k, l)] = max(0.0, max(vals) - data.tauV[(k, l)]) if vals else 0.0

    M: Dict[Tuple[int, int, int], float] = {}
    for (k, i, l) in F:
        M[(k, i, l)] = max(0.0, duration[(k, i, l)] - data.tauV[(k, l)])

    return F, W, M


# ============================================================
# AMPL model file writer (.mod) — bigM_n defined ONLY here
# ============================================================

def write_mod_file(mod_path: str = "van_robot_compact.mod") -> None:
    """Write the exact compact MILP formulation (identical to your DOcplex version)."""
    mod_content = r"""# van_robot_compact.mod
# Compact van-robot MILP - exact match to the original formulation
# (robust vs deterministic selected via precomputed rho / F / W / M in .dat)

param n integer > 0;

set V;
set A within V cross V;
set F within V cross V cross V;   # preprocessed feasible sorties (k,i,l)

param start := 0;
param end := n + 1;

param cV {A} >= 0;
param tauV {A} >= 0;
param sV {V} >= 0;
param cR {F} >= 0;
param rho {F} >= 0;          # sortie duration: nominal OR robust (chosen in Python)
param W {A} >= 0;
param M {F} >= 0;
param T {V} >= 0;
param H >= 0;
param gamma >= 0;
param lam >= 0;
param bigM_n := n;           # ← defined ONLY here

var x {A} binary;
var y {F} binary;
var t {V} >= 0;
var w {A} >= 0;
var f {A} >= 0;

minimize total_cost:
    sum {(k,l) in A} cV[k,l] * x[k,l]
  + sum {(k,i,l) in F} cR[k,i,l] * y[k,i,l]
  + gamma * sum {(k,l) in A} w[k,l]
  + lam * t[end];

# (6) serve once
s.t. serve_once {i in 1..n}:
    sum {(k,i) in A} x[k,i]
  + sum {(k,i,l) in F} y[k,i,l] = 1;

# (7) start once
s.t. start_departure_once:
    sum {(start,l) in A} x[start,l] = 1;

# (8) end once
s.t. end_arrival_once:
    sum {(k,end) in A} x[k,end] = 1;

# (9) van flow balance
s.t. van_flow_balance {i in 1..n}:
    sum {(k,i) in A} x[k,i] = sum {(i,l) in A} x[i,l];

# (10) at most one sortie per arc
s.t. one_sortie_per_arc {(k,l) in A}:
    sum {(k,i,l) in F} y[k,i,l] <= x[k,l];

# (11) scf depot flow
s.t. scf_depot_flow:
    sum {(start,l) in A} f[start,l]
  = sum {i in 1..n} sum {(k,i) in A} x[k,i];

# (12) scf balance at customers
s.t. scf_balance {i in 1..n}:
    sum {(k,i) in A} f[k,i] - sum {(i,l) in A} f[i,l]
  = sum {(k,i) in A} x[k,i];

# (13) scf capacity
s.t. scf_cap {(k,l) in A}:
    f[k,l] <= bigM_n * x[k,l];

# (14) start time
s.t. start_time_zero:
    t[start] = 0;

# (15) time upper bounds
s.t. time_upper {k in V}:
    t[k] <= T[k];

# (16) time zero if not visited
s.t. time_zero_if_not_visited {i in 1..n}:
    t[i] <= T[i] * sum {(k,i) in A} x[k,i];

# (17) horizon
s.t. route_horizon:
    t[end] <= H;

# (18) timing (Gurobi indicator constraint)
s.t. timing {(k,l) in A}:
    x[k,l] ==> (t[l] >= t[k] + sV[k] + tauV[k,l] + w[k,l]);

# (19) sync-wait lower bound
s.t. sync_wait {(k,i,l) in F}:
    w[k,l] >= rho[k,i,l] - tauV[k,l] - M[k,i,l] * (1 - y[k,i,l]);

# (20) wait upper bound
s.t. wait_upper {(k,l) in A}:
    w[k,l] <= W[k,l] * x[k,l];
"""
    os.makedirs(os.path.dirname(mod_path) or ".", exist_ok=True)
    with open(mod_path, "w") as f:
        f.write(mod_content)
    print(f"[INFO] AMPL model written: {mod_path}")


# ============================================================
# FIXED AMPL data writer — bigM_n line REMOVED (your latest preprocessing)
# ============================================================

def write_ampl_data_for_mode(
    data: VanRobotData,
    mode: str,
    dat_path: str,
) -> None:
    """Preprocess for the chosen mode and write clean .dat file."""
    validate_data_keys(data, strict=True)
    V, C, A, Fe, start, end = build_sets(data)

    if mode == "robust":
        sortie_duration = compute_robust_sortie_duration(data, Fe)
    else:
        sortie_duration = compute_nominal_sortie_duration(data, Fe)

    F, W, M = preprocess_sorties_from_duration(data, A, Fe, sortie_duration)

    os.makedirs(os.path.dirname(dat_path) or ".", exist_ok=True)
    with open(dat_path, "w") as f:
        f.write(f"param n := {data.n};\n\n")
        f.write("set V := " + " ".join(map(str, V)) + ";\n\n")

        f.write("set A :=\n")
        for k, l in A:
            f.write(f"  {k} {l}\n")
        f.write(";\n\n")

        f.write("set F :=\n")
        for k, i, l in F:
            f.write(f"  {k} {i} {l}\n")
        f.write(";\n\n")

        f.write("param cV :=\n")
        for k, l in A:
            f.write(f"  {k} {l} {data.cV[(k,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param tauV :=\n")
        for k, l in A:
            f.write(f"  {k} {l} {data.tauV[(k,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param sV :=\n")
        for k in V:
            f.write(f"  {k} {data.sV[k]:.10f}\n")
        f.write(";\n\n")

        f.write("param cR :=\n")
        for k, i, l in F:
            f.write(f"  {k} {i} {l} {data.cR[(k,i,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param rho :=\n")
        for k, i, l in F:
            f.write(f"  {k} {i} {l} {sortie_duration[(k,i,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param W :=\n")
        for k, l in A:
            f.write(f"  {k} {l} {W[(k,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param M :=\n")
        for k, i, l in F:
            f.write(f"  {k} {i} {l} {M[(k,i,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param T :=\n")
        for k in V:
            f.write(f"  {k} {data.T[k]:.10f}\n")
        f.write(";\n\n")

        f.write(f"param H := {data.H:.10f};\n")
        f.write(f"param gamma := {data.gamma:.10f};\n")
        f.write(f"param lam := {data.lam:.10f};\n")
        # bigM_n removed — defined only in .mod

    print(f"[INFO] {mode} .dat written → {dat_path}  (|F|={len(F)})")


# ============================================================
# Solve with AMPL + Gurobi (unchanged)
# ============================================================

def solve_model_ampl(
    data: VanRobotData,
    mode: str,
    instance_name: str,
    dat_dir: str = "outputs/dat",
    mod_path: str = "van_robot_compact.mod",
    time_limit: Optional[float] = 300.0,
    log_output: bool = False,
) -> Optional[AMPL]:
    """Solve one mode via AMPL/Gurobi. Returns AMPL object or None if infeasible."""
    if not os.path.exists(mod_path):
        write_mod_file(mod_path)

    dat_path = os.path.join(dat_dir, f"{instance_name}_{mode}.dat")
    write_ampl_data_for_mode(data, mode, dat_path)

    ampl = AMPL()
    ampl.setOption("solver", "gurobi")
    if time_limit is not None:
        ampl.setOption("gurobi_options", f"timelimit={time_limit} mipgap=1e-5")

    ampl.setOption("solver_msg", 1 if log_output else 0)

    ampl.read(mod_path)
    ampl.read_data(dat_path)

    start_t = time.time()
    ampl.solve()
    solve_time = time.time() - start_t

    status = ampl.solve_result
    if "solved" not in status.lower():
        print(f"[DEBUG] {mode} solve failed: {status} (time={solve_time:.2f}s)")
        return None

    print(f"[INFO] {mode} solved successfully (time={solve_time:.2f}s, obj={ampl.get_objective('total_cost').value():.4f})")
    return ampl


# ============================================================
# Solution extraction, objective breakdown, CSV rows, Monte Carlo (unchanged)
# ============================================================

# (All remaining functions are identical to your latest DOcplex version.
# They are included below for a self-contained notebook cell.)

def compute_objective_breakdown_ampl(ampl: AMPL, data: VanRobotData) -> Dict[str, float]:
    x_dict = ampl.get_variable("x").get_values().to_dict()
    y_dict = ampl.get_variable("y").get_values().to_dict()
    w_dict = ampl.get_variable("w").get_values().to_dict()
    t_dict = ampl.get_variable("t").get_values().to_dict()
    end = data.n + 1

    van_cost = sum(data.cV.get(a, 0.0) * x_dict.get(a, 0.0) for a in x_dict)
    robot_cost = sum(data.cR.get(s, 0.0) * y_dict.get(s, 0.0) for s in y_dict)
    waiting_cost = data.gamma * sum(w_dict.values())
    completion_cost = data.lam * t_dict[end]
    total = van_cost + robot_cost + waiting_cost + completion_cost

    return {
        "objective_total": total,
        "van_cost": van_cost,
        "robot_cost": robot_cost,
        "waiting_cost": waiting_cost,
        "completion_cost": completion_cost,
        "route_completion_time": t_dict[end],
        "total_waiting_time": sum(w_dict.values()),
    }


def extract_plan_ampl(ampl: AMPL, mode: str, data: VanRobotData) -> Dict[str, Any]:
    x_dict = ampl.get_variable("x").get_values().to_dict()
    y_dict = ampl.get_variable("y").get_values().to_dict()
    t_dict = ampl.get_variable("t").get_values().to_dict()
    w_dict = ampl.get_variable("w").get_values().to_dict()

    selected_arcs = [a for a, v in x_dict.items() if v > 0.5]
    selected_sorties = [s for s, v in y_dict.items() if v > 0.5]
    positive_wait = [(a, w_dict[a]) for a, v in w_dict.items() if v > 1e-8]
    arrival_times = t_dict

    V, C, A, Fe, _, _ = build_sets(data)
    if mode == "robust":
        sd = compute_robust_sortie_duration(data, Fe)
    else:
        sd = compute_nominal_sortie_duration(data, Fe)
    F, _, _ = preprocess_sorties_from_duration(data, A, Fe, sd)

    bd = compute_objective_breakdown_ampl(ampl, data)

    return {
        "mode": mode,
        "selected_arcs": selected_arcs,
        "selected_sorties": selected_sorties,
        "x": x_dict,
        "y": y_dict,
        "w": w_dict,
        "t": t_dict,
        "positive_wait": positive_wait,
        "arrival_times": arrival_times,
        "objective_breakdown": bd,
        "F": F,
        "Fe": list(Fe),
    }


def optimization_results_row_ampl(
    ampl: AMPL,
    data: VanRobotData,
    instance_name: str,
    seed: int,
    mode: str,
) -> Dict[str, object]:
    bd = compute_objective_breakdown_ampl(ampl, data)

    x_dict = ampl.get_variable("x").get_values().to_dict()
    y_dict = ampl.get_variable("y").get_values().to_dict()
    w_dict = ampl.get_variable("w").get_values().to_dict()
    selected_arcs = [a for a, v in x_dict.items() if v > 0.5]
    selected_sorties = [s for s, v in y_dict.items() if v > 0.5]
    positive_wait = [a for a, v in w_dict.items() if v > 1e-8]

    V, C, A, Fe, _, _ = build_sets(data)
    if mode == "robust":
        sd = compute_robust_sortie_duration(data, Fe)
    else:
        sd = compute_nominal_sortie_duration(data, Fe)
    F, _, _ = preprocess_sorties_from_duration(data, A, Fe, sd)

    status = ampl.solve_result
    try:
        solve_time = ampl.get_value("_solve_time")
    except:
        solve_time = None

    return {
        "instance": instance_name,
        "model_type": mode,
        "seed": seed,
        "n": data.n,
        "alpha": data.alpha,
        "num_arcs_A": len(A),
        "num_sorties_Fe": len(Fe),
        "num_sorties_F": len(F),
        "selected_van_arcs": len(selected_arcs),
        "selected_robot_sorties": len(selected_sorties),
        "positive_wait_arcs": len(positive_wait),
        "objective_total": round(bd["objective_total"], 6),
        "van_cost": round(bd["van_cost"], 6),
        "robot_cost": round(bd["robot_cost"], 6),
        "waiting_cost": round(bd["waiting_cost"], 6),
        "completion_cost": round(bd["completion_cost"], 6),
        "route_completion_time": round(bd["route_completion_time"], 6),
        "total_waiting_time": round(bd["total_waiting_time"], 6),
        "solve_status": status,
        "solve_time": solve_time,
        "mip_gap": None,
    }


def append_results_csv(csv_path: str, row: Dict[str, object]) -> None:
    os.makedirs(os.path.dirname(csv_path) or ".", exist_ok=True)
    exists = os.path.exists(csv_path)
    with open(csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def print_csv_style_row(row: Dict[str, object]) -> None:
    print(",".join(str(k) for k in row.keys()))
    print(",".join(str(v) for v in row.values()))


def sample_uniform_upward(nominal: float, delta: float, rng: random.Random) -> float:
    return nominal + rng.uniform(0.0, max(0.0, delta))


def reconstruct_route_from_selected_arcs(
    selected_arcs: List[Tuple[int, int]],
    start: int,
    end: int,
) -> List[int]:
    succ = {}
    for (k, l) in selected_arcs:
        succ[k] = l
    route = [start]
    cur = start
    seen = {start}
    while cur != end:
        if cur not in succ:
            raise ValueError(f"Cannot reconstruct route: no successor for node {cur}")
        cur = succ[cur]
        route.append(cur)
        if cur in seen and cur != end:
            raise ValueError("Cycle detected while reconstructing route")
        seen.add(cur)
        if len(route) > len(selected_arcs) + 2:
            raise ValueError("Route reconstruction exceeded expected length")
    return route


def simulate_plan_once(
    data: VanRobotData,
    plan: Dict[str, Any],
    rng: random.Random,
) -> Dict[str, Any]:
    V, C, A, Fe, start, end = build_sets(data)
    selected_arcs = list(plan["selected_arcs"])
    selected_sorties = list(plan["selected_sorties"])
    w_plan = plan["w"]

    route = reconstruct_route_from_selected_arcs(selected_arcs, start, end)

    t_sim = {k: None for k in V}
    t_sim[start] = 0.0

    for idx in range(len(route) - 1):
        k = route[idx]
        l = route[idx + 1]
        t_sim[l] = t_sim[k] + data.sV[k] + data.tauV[(k, l)] + w_plan.get((k, l), 0.0)

    failures = 0
    sampled_sortie_durations = {}

    for (k, i, l) in selected_sorties:
        tau_out_real = sample_uniform_upward(data.tauR_out[(k, i)], data.delta_out[(k, i)], rng)
        s_real = sample_uniform_upward(data.sR[i], data.delta_srv[i], rng)
        tau_in_real = sample_uniform_upward(data.tauR_in[(i, l)], data.delta_in[(i, l)], rng)

        robot_duration_real = tau_out_real + s_real + tau_in_real
        sampled_sortie_durations[(k, i, l)] = robot_duration_real

        robot_completion = t_sim[k] + data.sV[k] + robot_duration_real
        if robot_completion > t_sim[l] + 1e-9:
            failures += 1

    return {
        "trial_failed": 1 if failures > 0 else 0,
        "sync_failures": failures,
        "selected_sorties": len(selected_sorties),
        "route_completion_time": t_sim[end],
        "sampled_sortie_durations": sampled_sortie_durations,
    }


def monte_carlo_simulation(
    data: VanRobotData,
    plan: Dict[str, Any],
    num_trials: int = 1000,
    seed: int = 0,
) -> Dict[str, Any]:
    rng = random.Random(seed)
    trial_failures = 0
    sync_failures_total = 0
    completion_times: List[float] = []

    for _ in range(num_trials):
        res = simulate_plan_once(data, plan, rng)
        trial_failures += res["trial_failed"]
        sync_failures_total += res["sync_failures"]
        completion_times.append(res["route_completion_time"])

    reliability = 1.0 - trial_failures / max(1, num_trials)
    avg_sync_failures_per_trial = sync_failures_total / max(1, num_trials)
    avg_completion_time = sum(completion_times) / max(1, len(completion_times))

    return {
        "num_trials": num_trials,
        "trial_failures": trial_failures,
        "trial_failure_rate": trial_failures / max(1, num_trials),
        "reliability_no_sync_failure": reliability,
        "sync_failures_total": sync_failures_total,
        "avg_sync_failures_per_trial": avg_sync_failures_per_trial,
        "selected_sorties": len(plan["selected_sorties"]),
        "avg_route_completion_time_sim": avg_completion_time,
    }


def simulation_results_row(
    data: VanRobotData,
    instance_name: str,
    model_type: str,
    seed: int,
    sim_seed: int,
    sim_result: Dict[str, Any],
    opt_plan: Dict[str, Any],
) -> Dict[str, object]:
    bd = opt_plan["objective_breakdown"]
    return {
        "instance": instance_name,
        "model_type": model_type,
        "seed": seed,
        "sim_seed": sim_seed,
        "n": data.n,
        "alpha": data.alpha,
        "num_trials": sim_result["num_trials"],
        "selected_sorties": sim_result["selected_sorties"],
        "objective_total": round(bd["objective_total"], 6),
        "van_cost": round(bd["van_cost"], 6),
        "robot_cost": round(bd["robot_cost"], 6),
        "waiting_cost": round(bd["waiting_cost"], 6),
        "completion_cost": round(bd["completion_cost"], 6),
        "trial_failures": sim_result["trial_failures"],
        "trial_failure_rate": round(sim_result["trial_failure_rate"], 6),
        "reliability_no_sync_failure": round(sim_result["reliability_no_sync_failure"], 6),
        "sync_failures_total": sim_result["sync_failures_total"],
        "avg_sync_failures_per_trial": round(sim_result["avg_sync_failures_per_trial"], 6),
        "avg_route_completion_time_sim": round(sim_result["avg_route_completion_time_sim"], 6),
    }


# ============================================================
# Single-instance comparison (updated for AMPL)
# ============================================================

def solve_and_compare_on_instance(
    data: VanRobotData,
    instance_name: str,
    seed: int,
    sim_trials: int = 1000,
    sim_seed: int = 12345,
    time_limit: Optional[float] = 300.0,
    log_output: bool = False,
    data_dir: str = "outputs/dat",
    optimization_csv: str = "outputs/optimization_results.csv",
    simulation_csv: str = "outputs/simulation_results.csv",
) -> Dict[str, Any]:
    """Solve robust + deterministic via AMPL/Gurobi and run Monte Carlo."""
    os.makedirs(data_dir, exist_ok=True)

    results: Dict[str, Any] = {"instance": instance_name}

    # Robust
    ampl_rob = solve_model_ampl(
        data=data,
        mode="robust",
        instance_name=instance_name,
        dat_dir=data_dir,
        time_limit=time_limit,
        log_output=log_output,
    )
    if ampl_rob is not None:
        row = optimization_results_row_ampl(ampl_rob, data, instance_name, seed, "robust")
        append_results_csv(optimization_csv, row)

        plan_rob = extract_plan_ampl(ampl_rob, "robust", data)
        sim_rob = monte_carlo_simulation(data, plan_rob, num_trials=sim_trials, seed=sim_seed)
        sim_row_rob = simulation_results_row(
            data, instance_name, "robust", seed, sim_seed, sim_rob, plan_rob
        )
        append_results_csv(simulation_csv, sim_row_rob)
        results["robust_model"] = ampl_rob
        results["robust_plan"] = plan_rob
        results["robust_sim"] = sim_rob

    # Deterministic
    ampl_det = solve_model_ampl(
        data=data,
        mode="deterministic",
        instance_name=instance_name,
        dat_dir=data_dir,
        time_limit=time_limit,
        log_output=log_output,
    )
    if ampl_det is not None:
        row = optimization_results_row_ampl(ampl_det, data, instance_name, seed, "deterministic")
        append_results_csv(optimization_csv, row)

        plan_det = extract_plan_ampl(ampl_det, "deterministic", data)
        sim_det = monte_carlo_simulation(data, plan_det, num_trials=sim_trials, seed=sim_seed)
        sim_row_det = simulation_results_row(
            data, instance_name, "deterministic", seed, sim_seed, sim_det, plan_det
        )
        append_results_csv(simulation_csv, sim_row_det)
        results["deterministic_model"] = ampl_det
        results["deterministic_plan"] = plan_det
        results["deterministic_sim"] = sim_det

    if "robust_sim" in results and "deterministic_sim" in results:
        rob_rel = results["robust_sim"]["reliability_no_sync_failure"]
        det_rel = results["deterministic_sim"]["reliability_no_sync_failure"]
        results["reliability_gap_robust_minus_deterministic"] = rob_rel - det_rel

    return results


# ============================================================
# Batch runner (unchanged)
# ============================================================

def run_batch_experiments(
    n_values: List[int],
    alpha_values: List[float],
    seeds: List[int],
    sim_trials: int = 1000,
    sim_seed_offset: int = 100000,
    time_limit: Optional[float] = 300.0,
    log_output: bool = False,
    optimization_csv: str = "outputs/optimization_results.csv",
    simulation_csv: str = "outputs/simulation_results.csv",
    data_dir: str = "outputs/dat",
) -> List[Dict[str, Any]]:
    all_results: List[Dict[str, Any]] = []

    for n in n_values:
        for alpha in alpha_values:
            for seed in seeds:
                instance_name = f"euclidean_n{n}_a{int(round(alpha * 100)):03d}_s{seed}"
                print(f"\n[INFO] Running instance {instance_name}")

                data = generate_random_euclidean_instance(n=n, alpha=alpha, seed=seed)

                res = solve_and_compare_on_instance(
                    data=data,
                    instance_name=instance_name,
                    seed=seed,
                    sim_trials=sim_trials,
                    sim_seed=sim_seed_offset + seed,
                    time_limit=time_limit,
                    log_output=log_output,
                    data_dir=data_dir,
                    optimization_csv=optimization_csv,
                    simulation_csv=simulation_csv,
                )

                all_results.append(res)

                if "robust_sim" in res:
                    print("[INFO] Robust reliability:", round(res["robust_sim"]["reliability_no_sync_failure"], 6))
                if "deterministic_sim" in res:
                    print("[INFO] Deterministic reliability:", round(res["deterministic_sim"]["reliability_no_sync_failure"], 6))
                if "reliability_gap_robust_minus_deterministic" in res:
                    print("[INFO] Reliability gap (robust - deterministic):",
                          round(res["reliability_gap_robust_minus_deterministic"], 6))

    return all_results


# ============================================================
# Main examples (run this cell after the two setup cells)
# ============================================================

if __name__ == "__main__" or "get_ipython" in globals():
    write_mod_file("van_robot_compact.mod")

    data = generate_random_euclidean_instance(
    n=12,
    alpha=0.20,
    seed=1,
    area_size=180.0,
    van_speed=1.0,
    robot_speed=1.6,
    van_cost_per_distance=1.0,
    robot_cost_per_distance=0.2,
    customer_service_time_van=4.0,
    customer_service_time_robot=0.6,
    robot_endurance_factor=1.2,
    gamma=0.08,
    lam=0.01,
    horizon_factor=2.0,
)
    comparison = solve_and_compare_on_instance(
        data=data,
        instance_name="euclidean_n10_a010_s42",
        seed=42,
        sim_trials=1000,
        sim_seed=12345,
        time_limit=300.0,
        log_output=False,
        data_dir="outputs/dat",
        optimization_csv="outputs/optimization_results.csv",
        simulation_csv="outputs/simulation_results.csv",
    )

    if "robust_sim" in comparison:
        print("\n[INFO] Robust simulation summary:")
        print_csv_style_row({
            "instance": "euclidean_n10_a010_s42",
            "model_type": "robust",
            "reliability_no_sync_failure": round(comparison["robust_sim"]["reliability_no_sync_failure"], 6),
            "trial_failure_rate": round(comparison["robust_sim"]["trial_failure_rate"], 6),
            "sync_failures_total": comparison["robust_sim"]["sync_failures_total"],
        })

    if "deterministic_sim" in comparison:
        print("\n[INFO] Deterministic simulation summary:")
        print_csv_style_row({
            "instance": "euclidean_n10_a010_s42",
            "model_type": "deterministic",
            "reliability_no_sync_failure": round(comparison["deterministic_sim"]["reliability_no_sync_failure"], 6),
            "trial_failure_rate": round(comparison["deterministic_sim"]["trial_failure_rate"], 6),
            "sync_failures_total": comparison["deterministic_sim"]["sync_failures_total"],
        })

## AMPL Model Files

Write the `.mod` and `.dat` template files to disk for use by the solver.

In [ ]:
%%writefile van_robot_compact_from_coords.mod
param n integer > 0;

set C := 1..n;
param start integer := 0;
param finish integer := n + 1;
set V := 0..n+1;

param use_robust integer default 1;

param xcoord {V};
param ycoord {V};

param van_speed > 0;
param robot_speed > 0;
param van_cost_per_distance >= 0;
param robot_cost_per_distance >= 0;
param service_time_van >= 0 default 2.0;
param service_time_robot >= 0 default 1.0;
param alpha >= 0, <= 1;
param robot_endurance_factor > 0;
param gamma > 0;
param lam >= 0;
param horizon_factor > 0;

param sV_raw {C} default service_time_van;
param sR_raw {C} default service_time_robot;

set A  := setof {k in V, l in V: k != l and l != start and k != finish} (k,l);
set KOI := setof {k in V, i in C: i != k} (k,i);
set IOL := setof {i in C, l in V: i != l} (i,l);
set Fe := setof {(k,l) in A, i in C: i != k and i != l} (k,i,l);

param distV {(k,l) in A} :=
    sqrt((xcoord[k] - xcoord[l])^2 + (ycoord[k] - ycoord[l])^2);

param tauV {(k,l) in A} := distV[k,l] / van_speed;
param cV   {(k,l) in A} := van_cost_per_distance * distV[k,l];

param tauR_out {(k,i) in KOI} :=
    sqrt((xcoord[k] - xcoord[i])^2 + (ycoord[k] - ycoord[i])^2) / robot_speed;

param tauR_in {(i,l) in IOL} :=
    sqrt((xcoord[i] - xcoord[l])^2 + (ycoord[i] - ycoord[l])^2) / robot_speed;

param sV {k in V} := if k in C then sV_raw[k] else 0;
param sR {i in C} := sR_raw[i];

param delta_out {(k,i) in KOI} := alpha * tauR_out[k,i];
param delta_in  {(i,l) in IOL} := alpha * tauR_in[i,l];
param delta_srv {i in C} := alpha * sR[i];

param cR {(k,i,l) in Fe} := robot_cost_per_distance *
    (
      sqrt((xcoord[k] - xcoord[i])^2 + (ycoord[k] - ycoord[i])^2)
    + sqrt((xcoord[i] - xcoord[l])^2 + (ycoord[i] - ycoord[l])^2)
    );

param rho_nom {(k,i,l) in Fe} := tauR_out[k,i] + sR[i] + tauR_in[i,l];
param rho_rob {(k,i,l) in Fe} := rho_nom[k,i,l] + delta_out[k,i] + delta_srv[i] + delta_in[i,l];
param rho {(k,i,l) in Fe} := if use_robust = 1 then rho_rob[k,i,l] else rho_nom[k,i,l];

param avg_nominal_sortie :=
    if card(Fe) > 0 then (sum {(k,i,l) in Fe} rho_nom[k,i,l]) / card(Fe) else 0;

param E_R := robot_endurance_factor * avg_nominal_sortie;

param avg_tauV :=
    if card(A) > 0 then (sum {(k,l) in A} tauV[k,l]) / card(A) else 0;

param H := horizon_factor * ((n + 1) * avg_tauV + sum {i in C} sV[i] + E_R);

param T {k in V} := H;

set F := setof {(k,i,l) in Fe: rho[k,i,l] <= E_R} (k,i,l);

param raw_wait {(k,l) in A} :=
    if card(setof {i in C: (k,i,l) in F} i) > 0 then
        max {i in C: (k,i,l) in F} (rho[k,i,l] - tauV[k,l])
    else 0;

param W {(k,l) in A} := if raw_wait[k,l] > 0 then raw_wait[k,l] else 0;
param M {(k,i,l) in F} := if rho[k,i,l] - tauV[k,l] > 0 then rho[k,i,l] - tauV[k,l] else 0;

param bigM_n := n;

var x {A} binary;
var y {F} binary;
var t {V} >= 0;
var w {A} >= 0;
var f {A} >= 0;

minimize total_cost:
    sum {(k,l) in A} cV[k,l] * x[k,l]
  + sum {(k,i,l) in F} cR[k,i,l] * y[k,i,l]
  + gamma * sum {(k,l) in A} w[k,l]
  + lam * t[finish];

s.t. serve_once {i in C}:
    sum {(k,i) in A} x[k,i]
  + sum {(k,i,l) in F} y[k,i,l] = 1;

s.t. start_departure_once:
    sum {(start,l) in A} x[start,l] = 1;

s.t. end_arrival_once:
    sum {(k,finish) in A} x[k,finish] = 1;

s.t. van_flow_balance {i in C}:
    sum {(k,i) in A} x[k,i] = sum {(i,l) in A} x[i,l];

s.t. one_sortie_per_arc {(k,l) in A}:
    sum {(k,i,l) in F} y[k,i,l] <= x[k,l];

s.t. scf_depot_flow:
    sum {(start,l) in A} f[start,l]
  = sum {i in C} sum {(k,i) in A} x[k,i];

s.t. scf_balance {i in C}:
    sum {(k,i) in A} f[k,i] - sum {(i,l) in A} f[i,l]
  = sum {(k,i) in A} x[k,i];

s.t. scf_cap {(k,l) in A}:
    f[k,l] <= bigM_n * x[k,l];

s.t. start_time_zero:
    t[start] = 0;

s.t. time_upper {k in V}:
    t[k] <= T[k];

s.t. time_zero_if_not_visited {i in C}:
    t[i] <= T[i] * sum {(k,i) in A} x[k,i];

s.t. route_horizon:
    t[finish] <= H;

s.t. timing {(k,l) in A}:
    x[k,l] ==> (t[l] >= t[k] + sV[k] + tauV[k,l] + w[k,l]);

s.t. sync_wait {(k,i,l) in F}:
    w[k,l] >= rho[k,i,l] - tauV[k,l] - M[k,i,l] * (1 - y[k,i,l]);

s.t. wait_upper {(k,l) in A}:
    w[k,l] <= W[k,l] * x[k,l];

In [ ]:
%%writefile instance_template.dat
param n := 6;

param xcoord :=
0  0
1  10
2  22
3  35
4  48
5  60
6  73
7  90
;

param ycoord :=
0  0
1  18
2  42
3  15
4  36
5  10
6  28
7  0
;

param van_speed := 1.0;
param robot_speed := 1.4;
param van_cost_per_distance := 1.0;
param robot_cost_per_distance := 0.20;
param service_time_van := 4.0;
param service_time_robot := 0.6;
param alpha := 0.10;
param robot_endurance_factor := 1.20;
param gamma := 0.08;
param lam := 0.01;
param horizon_factor := 2.0;

In [ ]:
from amplpy import AMPL

ampl = AMPL()
ampl.setOption("solver", "gurobi")
ampl.setOption("gurobi_options", "timelimit=300 mipgap=1e-5")

ampl.read("van_robot_compact_from_coords.mod")
ampl.readData("instance_template.dat")

ampl.solve()

print(ampl.getObjective("total_cost").value())

---
## Week 1 — Deterministic Baseline

Solve the deterministic MILP for a single instance and a batch grid.
Functions here are specific to the Week 1 `.mod` formulation.

In [ ]:
def reconstruct_route(selected_arcs, start, end):
    succ = {}
    for (k, l) in selected_arcs:
        if k in succ:
            raise ValueError(
                f"Route reconstruction error: node {k} has multiple successors "
                f"{succ[k]} and {l}"
            )
        succ[k] = l

    route = [start]
    cur = start
    seen = {start}

    while cur != end:
        if cur not in succ:
            raise ValueError(f"Cannot reconstruct route: no successor for node {cur}")
        cur = succ[cur]
        route.append(cur)
        if cur in seen and cur != end:
            raise ValueError("Cycle detected in route reconstruction")
        seen.add(cur)
        if len(route) > len(selected_arcs) + 2:
            raise ValueError("Route reconstruction exceeded expected length")

    return route


def validate_route_coverage(route, selected_arcs, start, end):
    if not route:
        raise ValueError("Empty reconstructed route")
    if route[0] != start:
        raise ValueError(f"Route does not start at {start}")
    if route[-1] != end:
        raise ValueError(f"Route does not end at {end}")

    route_arcs = {(route[i], route[i + 1]) for i in range(len(route) - 1)}
    extra_arcs = set(selected_arcs) - route_arcs
    if extra_arcs:
        raise ValueError(f"Selected arcs not on reconstructed main route: {sorted(extra_arcs)}")

In [ ]:
def write_det_mod_file(mod_path: str = "week1_det.mod") -> None:
    mod_text = r"""
param n integer > 0;

set V;
set A within V cross V;
set F within V cross V cross V;

param start := 0;
param end := n + 1;

param cV {A} >= 0;
param tauV {A} >= 0;
param sV {V} >= 0;

param cR {F} >= 0;
param rho {F} >= 0;
param W {A} >= 0;
param M {F} >= 0;

param T {V} >= 0;
param H >= 0;
param gamma >= 0;
param lam >= 0;

param bigM_n := n;

var x {A} binary;
var y {F} binary;
var t {V} >= 0;
var w {A} >= 0;
var f {A} >= 0;

minimize total_cost:
    sum {(k,l) in A} cV[k,l] * x[k,l]
  + sum {(k,i,l) in F} cR[k,i,l] * y[k,i,l]
  + gamma * sum {(k,l) in A} w[k,l]
  + lam * t[end];

s.t. serve_once {i in 1..n}:
    sum {(k,i) in A} x[k,i]
  + sum {(k,i,l) in F} y[k,i,l] = 1;

s.t. start_departure_once:
    sum {(start,l) in A} x[start,l] = 1;

s.t. end_arrival_once:
    sum {(k,end) in A} x[k,end] = 1;

s.t. van_flow_balance {i in 1..n}:
    sum {(k,i) in A} x[k,i] = sum {(i,l) in A} x[i,l];

s.t. one_sortie_per_arc {(k,l) in A}:
    sum {(k,i,l) in F} y[k,i,l] <= x[k,l];

s.t. scf_depot_flow:
    sum {(start,l) in A} f[start,l]
  = sum {i in 1..n} sum {(k,i) in A} x[k,i];

s.t. scf_balance {i in 1..n}:
    sum {(k,i) in A} f[k,i] - sum {(i,l) in A} f[i,l]
  = sum {(k,i) in A} x[k,i];

s.t. scf_cap {(k,l) in A}:
    f[k,l] <= bigM_n * x[k,l];

s.t. start_time_zero:
    t[start] = 0;

s.t. time_upper {k in V}:
    t[k] <= T[k];

s.t. time_zero_if_not_visited {i in 1..n}:
    t[i] <= T[i] * sum {(k,i) in A} x[k,i];

s.t. route_horizon:
    t[end] <= H;

s.t. timing {(k,l) in A}:
    x[k,l] ==> (t[l] >= t[k] + sV[k] + tauV[k,l] + w[k,l]);

s.t. sync_wait {(k,i,l) in F}:
    w[k,l] >= rho[k,i,l] - tauV[k,l] - M[k,i,l] * (1 - y[k,i,l]);

s.t. wait_upper {(k,l) in A}:
    w[k,l] <= W[k,l] * x[k,l];
"""
    Path(mod_path).write_text(mod_text.strip() + "\n")
    print(f"✅ wrote {mod_path}")

In [ ]:
def write_det_dat_file(data: VanRobotData, dat_path: str = "week1_det.dat") -> Dict[str, Any]:
    validate_data_keys(data, strict=True)

    V, C, A, Fe, start, end = build_sets(data)
    F, rho_nom, W, M = preprocess_deterministic_sorties(data, A, Fe)

    with open(dat_path, "w") as f:
        f.write(f"param n := {data.n};\n\n")

        f.write("set V := " + " ".join(map(str, V)) + ";\n\n")

        f.write("set A :=\n")
        for (k, l) in A:
            f.write(f"  {k} {l}\n")
        f.write(";\n\n")

        f.write("set F :=\n")
        for (k, i, l) in F:
            f.write(f"  {k} {i} {l}\n")
        f.write(";\n\n")

        f.write("param cV :=\n")
        for (k, l) in A:
            f.write(f"  {k} {l} {data.cV[(k,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param tauV :=\n")
        for (k, l) in A:
            f.write(f"  {k} {l} {data.tauV[(k,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param sV :=\n")
        for k in V:
            f.write(f"  {k} {data.sV[k]:.10f}\n")
        f.write(";\n\n")

        f.write("param cR :=\n")
        for (k, i, l) in F:
            f.write(f"  {k} {i} {l} {data.cR[(k,i,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param rho :=\n")
        for (k, i, l) in F:
            f.write(f"  {k} {i} {l} {rho_nom[(k,i,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param W :=\n")
        for (k, l) in A:
            f.write(f"  {k} {l} {W[(k,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param M :=\n")
        for (k, i, l) in F:
            f.write(f"  {k} {i} {l} {M[(k,i,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param T :=\n")
        for k in V:
            f.write(f"  {k} {data.T[k]:.10f}\n")
        f.write(";\n\n")

        f.write(f"param H := {data.H:.10f};\n")
        f.write(f"param gamma := {data.gamma:.10f};\n")
        f.write(f"param lam := {data.lam:.10f};\n")

    info = {
        "num_arcs_A": len(A),
        "num_candidate_sorties_Fe": len(Fe),
        "num_feasible_sorties_F": len(F),
        "dat_path": dat_path,
    }
    print(f"✅ wrote {dat_path} | |A|={len(A)} |Fe|={len(Fe)} |F|={len(F)}")
    return info

In [ ]:
def solve_deterministic_baseline_ampl(
    data: VanRobotData,
    mod_path: str = "week1_det.mod",
    dat_path: str = "week1_det.dat",
    time_limit: float = 30.0,
    log_to_console: bool = True,
) -> Dict[str, Any]:
    write_det_mod_file(mod_path)
    info = write_det_dat_file(data, dat_path)

    local_ampl = AMPL()
    local_ampl.setOption("solver", "gurobi")
    local_ampl.setOption("solver_msg", 1 if log_to_console else 0)
    local_ampl.setOption("gurobi_options", f"timelimit={time_limit} mipgap=1e-5")

    local_ampl.read(mod_path)
    local_ampl.read_data(dat_path)

    wall_start = time.time()
    local_ampl.solve()
    wall_runtime = time.time() - wall_start

    status = local_ampl.solve_result

    if "solved" not in status.lower():
        return {
            "status": status,
            "runtime": wall_runtime,
            "objective": None,
            "route": None,
            "sorties": [],
            "waiting": [],
            "completion_time": None,
            "num_feasible_sorties": info["num_feasible_sorties_F"],
            "objective_breakdown": None,
        }

    x_vals = local_ampl.get_variable("x").get_values().to_dict()
    y_vals = local_ampl.get_variable("y").get_values().to_dict()
    w_vals = local_ampl.get_variable("w").get_values().to_dict()
    t_vals = local_ampl.get_variable("t").get_values().to_dict()

    V, C, A, Fe, start, end = build_sets(data)

    selected_arcs = sorted([a for a, v in x_vals.items() if v > 0.5])
    selected_sorties = sorted([s for s, v in y_vals.items() if v > 0.5])
    positive_wait = sorted([(a, round(v, 6)) for a, v in w_vals.items() if v > 1e-8])

    route = reconstruct_route(selected_arcs, start, end)
    validate_route_coverage(route, selected_arcs, start, end)

    van_cost = sum(data.cV[a] * x_vals.get(a, 0.0) for a in A)
    robot_cost = sum(data.cR[s] * y_vals.get(s, 0.0) for s in y_vals)
    waiting_cost = data.gamma * sum(w_vals.values())
    completion_cost = data.lam * t_vals[end]

    return {
        "status": status,
        "runtime": wall_runtime,
        "objective": local_ampl.get_objective("total_cost").value(),
        "route": route,
        "sorties": selected_sorties,
        "waiting": positive_wait,
        "completion_time": t_vals[end],
        "num_feasible_sorties": info["num_feasible_sorties_F"],
        "objective_breakdown": {
            "van_cost": van_cost,
            "robot_cost": robot_cost,
            "waiting_cost": waiting_cost,
            "completion_cost": completion_cost,
        },
        "ampl": local_ampl,
    }

In [ ]:
def run_week1_batch_ampl(
    n_values: List[int],
    alpha: float,
    seeds: List[int],
    time_limit: float = 30.0,
    output_csv: str = "week1_deterministic_results_ampl.csv",
) -> List[Dict[str, Any]]:
    results: List[Dict[str, Any]] = []

    fieldnames = [
        "n", "alpha", "seed", "status", "objective", "route", "sorties",
        "waiting", "completion_time", "runtime", "num_feasible_sorties",
        "van_cost", "robot_cost", "waiting_cost", "completion_cost"
    ]

    with open(output_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for n in n_values:
            for seed in seeds:
                data = generate_random_euclidean_instance(
                    n=n,
                    alpha=alpha,
                    seed=seed,
                    area_size=120.0,
                    van_speed=1.0,
                    robot_speed=1.3,
                    van_cost_per_distance=1.0,
                    robot_cost_per_distance=0.4,
                    customer_service_time_van=3.0,
                    customer_service_time_robot=1.0,
                    robot_endurance_factor=1.5,
                    gamma=0.10,
                    lam=0.01,
                    horizon_factor=2.0,
                )

                res = solve_deterministic_baseline_ampl(
                    data=data,
                    mod_path=f"week1_det_n{n}_s{seed}.mod",
                    dat_path=f"week1_det_n{n}_s{seed}.dat",
                    time_limit=time_limit,
                    log_to_console=False,
                )

                bd = res["objective_breakdown"] or {
                    "van_cost": None,
                    "robot_cost": None,
                    "waiting_cost": None,
                    "completion_cost": None,
                }

                row = {
                    "n": n,
                    "alpha": alpha,
                    "seed": seed,
                    "status": res["status"],
                    "objective": round(res["objective"], 6) if res["objective"] is not None else None,
                    "route": str(res["route"]),
                    "sorties": str(res["sorties"]),
                    "waiting": str(res["waiting"]),
                    "completion_time": round(res["completion_time"], 6) if res["completion_time"] is not None else None,
                    "runtime": round(res["runtime"], 6),
                    "num_feasible_sorties": res["num_feasible_sorties"],
                    "van_cost": round(bd["van_cost"], 6) if bd["van_cost"] is not None else None,
                    "robot_cost": round(bd["robot_cost"], 6) if bd["robot_cost"] is not None else None,
                    "waiting_cost": round(bd["waiting_cost"], 6) if bd["waiting_cost"] is not None else None,
                    "completion_cost": round(bd["completion_cost"], 6) if bd["completion_cost"] is not None else None,
                }

                writer.writerow(row)
                results.append(row)

                print(f"\nDeterministic baseline (AMPL) | n={n}, alpha={alpha}, seed={seed}")
                print("status:", row["status"])
                print("objective:", row["objective"])
                print("route:", row["route"])
                print("sorties:", row["sorties"])
                print("waiting:", row["waiting"])
                print("completion time:", row["completion_time"])
                print("runtime:", row["runtime"])
                print("num feasible sorties:", row["num_feasible_sorties"])
                print("objective breakdown:", {
                    "van_cost": row["van_cost"],
                    "robot_cost": row["robot_cost"],
                    "waiting_cost": row["waiting_cost"],
                    "completion_cost": row["completion_cost"],
                })

    print(f"\nSaved Week 1 AMPL results to: {output_csv}")
    return results

### Run — Single Instance

In [ ]:
data = generate_random_euclidean_instance(
    n=5,
    alpha=0.10,
    seed=42,
    area_size=120.0,
    van_speed=1.0,
    robot_speed=1.3,
    van_cost_per_distance=1.0,
    robot_cost_per_distance=0.4,
    customer_service_time_van=3.0,
    customer_service_time_robot=1.0,
    robot_endurance_factor=1.5,
    gamma=0.10,
    lam=0.01,
    horizon_factor=2.0,
)

res = solve_deterministic_baseline_ampl(
    data=data,
    mod_path="week1_det.mod",
    dat_path="week1_det.dat",
    time_limit=30.0,
    log_to_console=True,
)

print("\n=== WEEK 1 DETERMINISTIC BASELINE (AMPL) ===")
print("status:", res["status"])
print("objective:", round(res["objective"], 6) if res["objective"] is not None else None)
print("route:", res["route"])
print("sorties:", res["sorties"])
print("waiting:", res["waiting"])
print("completion time:", round(res["completion_time"], 6) if res["completion_time"] is not None else None)
print("runtime:", round(res["runtime"], 6))
print("num feasible sorties:", res["num_feasible_sorties"])
print("objective breakdown:", {
    k: round(v, 6) for k, v in res["objective_breakdown"].items()
})

In [ ]:
write_mod_file("van_robot_compact.mod")

data = generate_random_euclidean_instance(
    n=10,
    alpha=0.10,
    seed=42,
    area_size=120.0,
    van_speed=1.0,
    robot_speed=1.3,
    van_cost_per_distance=1.0,
    robot_cost_per_distance=0.4,
    customer_service_time_van=3.0,
    customer_service_time_robot=1.0,
    robot_endurance_factor=1.5,
    gamma=0.10,
    lam=0.01,
    horizon_factor=2.0,
)

write_dat(data, "instance_robust.dat", mode="robust")

ampl_rob = solve_model_ampl(
    mod_path="van_robot_compact.mod",
    dat_path="instance_robust.dat",
    time_limit=300.0,
    show_output=True
)

In [ ]:
write_dat(data, "instance_deterministic.dat", mode="deterministic")

ampl_det = solve_model_ampl(
    mod_path="van_robot_compact.mod",
    dat_path="instance_deterministic.dat",
    time_limit=300.0,
    show_output=True
)

sol_det = extract_solution(ampl_det)

print("\nDeterministic selected van arcs:")
for arc in sol_det["selected_arcs"]:
    print(arc)

print("\nDeterministic selected robot sorties:")
for sortie in sol_det["selected_sorties"]:
    print(sortie)

In [ ]:
obj_rob = ampl_rob.get_objective("total_cost").value()
obj_det = ampl_det.get_objective("total_cost").value()

print("Robust objective       =", round(obj_rob, 6))
print("Deterministic objective =", round(obj_det, 6))
print("Difference              =", round(obj_rob - obj_det, 6))

In [ ]:
print("Robust objective       =", round(ampl_rob.get_objective("total_cost").value(), 6))
print("Deterministic objective =", round(ampl_det.get_objective("total_cost").value(), 6))

w_rob = ampl_rob.get_variable("w").get_values().to_dict()
w_det = ampl_det.get_variable("w").get_values().to_dict()

total_wait_rob = sum(w_rob.values())
total_wait_det = sum(w_det.values())

print("Robust total waiting       =", round(total_wait_rob, 6))
print("Deterministic total waiting =", round(total_wait_det, 6))

y_rob = ampl_rob.get_variable("y").get_values().to_dict()
y_det = ampl_det.get_variable("y").get_values().to_dict()

num_sorties_rob = sum(1 for v in y_rob.values() if v > 0.5)
num_sorties_det = sum(1 for v in y_det.values() if v > 0.5)

print("Robust selected robot sorties       =", num_sorties_rob)
print("Deterministic selected robot sorties =", num_sorties_det)

In [ ]:
# objective breakdown
obj_rob = ampl_rob.get_objective("total_cost").value()
obj_det = ampl_det.get_objective("total_cost").value()

t_rob = ampl_rob.get_variable("t").get_values().to_dict()
t_det = ampl_det.get_variable("t").get_values().to_dict()

x_rob = ampl_rob.get_variable("x").get_values().to_dict()
x_det = ampl_det.get_variable("x").get_values().to_dict()

y_rob = ampl_rob.get_variable("y").get_values().to_dict()
y_det = ampl_det.get_variable("y").get_values().to_dict()

end_node = data.n + 1

rob_arcs = sorted([k for k, v in x_rob.items() if v > 0.5])
det_arcs = sorted([k for k, v in x_det.items() if v > 0.5])

rob_sorties = sorted([k for k, v in y_rob.items() if v > 0.5])
det_sorties = sorted([k for k, v in y_det.items() if v > 0.5])

print("Robust completion time       =", round(t_rob[end_node], 6))
print("Deterministic completion time =", round(t_det[end_node], 6))
print("Completion time difference    =", round(t_rob[end_node] - t_det[end_node], 6))

print("\nRobust van arcs:")
print(rob_arcs)

print("\nDeterministic van arcs:")
print(det_arcs)

print("\nSame van arcs?", rob_arcs == det_arcs)

print("\nRobust robot sorties:")
print(rob_sorties)

print("\nDeterministic robot sorties:")
print(det_sorties)

print("\nSame robot sorties?", rob_sorties == det_sorties)

### Run — Batch Experiments

In [ ]:
batch_results = run_week1_batch_ampl(
    n_values=[5, 8, 10],
    alpha=0.10,
    seeds=[1, 2, 42],
    time_limit=30.0,
    output_csv="week1_deterministic_results_ampl.csv",
)

In [ ]:
batch_results = run_batch_experiments(
    n_values=[8, 10, 12],
    alpha_values=[0.05, 0.10, 0.20],
    seeds=[1, 2, 3],
    sim_trials=1000,
    time_limit=300.0,
    data_dir="outputs/dat",
)

---
## Week 2 — Deterministic Simulation

Monte-Carlo stress testing of the deterministic plan under realized travel-time noise.

In [ ]:
import ast
import csv
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any


# ============================================
# Data container
# ============================================

@dataclass
class VanRobotData:
    n: int
    cV: Dict[Tuple[int, int], float]
    tauV: Dict[Tuple[int, int], float]
    sV: Dict[int, float]
    tauR_out: Dict[Tuple[int, int], float]
    sR: Dict[int, float]
    tauR_in: Dict[Tuple[int, int], float]
    cR: Dict[Tuple[int, int, int], float]
    delta_out: Dict[Tuple[int, int], float]
    delta_srv: Dict[int, float]
    delta_in: Dict[Tuple[int, int], float]
    T: Dict[int, float]
    E_R: float
    H: float
    gamma: float
    lam: float
    coords: Optional[Dict[int, Tuple[float, float]]] = None
    alpha: Optional[float] = None


# ============================================
# Sets and instance generation
# ============================================

def build_sets(n: int):
    start = 0
    end = n + 1
    C = list(range(1, n + 1))
    V = [start] + C + [end]
    A = [(k, l) for k in V for l in V if k != l and l != start and k != end]
    Fe = [(k, i, l) for (k, l) in A for i in C if i != k and i != l]
    return V, C, A, Fe, start, end


def euclidean(p: Tuple[float, float], q: Tuple[float, float]) -> float:
    return math.hypot(p[0] - q[0], p[1] - q[1])


def generate_random_euclidean_instance(
    n: int,
    alpha: float,
    seed: int = 0,
    area_size: float = 120.0,
    van_speed: float = 1.0,
    robot_speed: float = 1.3,
    van_cost_per_distance: float = 1.0,
    robot_cost_per_distance: float = 0.4,
    customer_service_time_van: float = 3.0,
    customer_service_time_robot: float = 1.0,
    robot_endurance_factor: float = 1.5,
    gamma: float = 0.10,
    lam: float = 0.01,
    horizon_factor: float = 2.0,
) -> VanRobotData:
    allowed_alphas = {0.05, 0.10, 0.20}
    if alpha not in allowed_alphas:
        raise ValueError(f"alpha must be one of {sorted(allowed_alphas)}, got {alpha}")

    rng = random.Random(seed)
    V, C, A, Fe, start, end = build_sets(n)

    coords = {k: (rng.uniform(0.0, area_size), rng.uniform(0.0, area_size)) for k in V}

    cV = {}
    tauV = {}
    for (k, l) in A:
        d = euclidean(coords[k], coords[l])
        cV[(k, l)] = van_cost_per_distance * d
        tauV[(k, l)] = d / van_speed

    sV = {k: customer_service_time_van for k in V}
    sV[start] = 0.0
    sV[end] = 0.0

    tauR_out = {}
    tauR_in = {}
    delta_out = {}
    delta_in = {}
    sR = {i: customer_service_time_robot for i in C}
    delta_srv = {i: alpha * sR[i] for i in C}

    for k in V:
        for i in C:
            if i != k:
                d = euclidean(coords[k], coords[i])
                tauR_out[(k, i)] = d / robot_speed
                delta_out[(k, i)] = alpha * tauR_out[(k, i)]

    for i in C:
        for l in V:
            if i != l:
                d = euclidean(coords[i], coords[l])
                tauR_in[(i, l)] = d / robot_speed
                delta_in[(i, l)] = alpha * tauR_in[(i, l)]

    cR = {}
    nominal_sortie_durations = []
    for (k, l) in A:
        for i in C:
            if i != k and i != l:
                sortie_dist = euclidean(coords[k], coords[i]) + euclidean(coords[i], coords[l])
                cR[(k, i, l)] = robot_cost_per_distance * sortie_dist
                nominal_sortie_durations.append(
                    tauR_out[(k, i)] + sR[i] + tauR_in[(i, l)]
                )

    avg_nominal_sortie = sum(nominal_sortie_durations) / max(1, len(nominal_sortie_durations))
    E_R = robot_endurance_factor * avg_nominal_sortie

    avg_tauV = sum(tauV.values()) / max(1, len(tauV))
    H = horizon_factor * ((n + 1) * avg_tauV + sum(sV[k] for k in C) + E_R)
    if H <= 0:
        H = 1000.0

    T = {k: H for k in V}

    return VanRobotData(
        n=n,
        cV=cV,
        tauV=tauV,
        sV=sV,
        tauR_out=tauR_out,
        sR=sR,
        tauR_in=tauR_in,
        cR=cR,
        delta_out=delta_out,
        delta_srv=delta_srv,
        delta_in=delta_in,
        T=T,
        E_R=E_R,
        H=H,
        gamma=gamma,
        lam=lam,
        coords=coords,
        alpha=alpha,
    )


# ============================================
# Parsing Week 1 CSV
# ============================================

def safe_literal_eval(x):
    if x is None:
        return None
    if isinstance(x, (list, tuple, dict, int, float)):
        return x
    s = str(x).strip()
    if s == "" or s.lower() == "none":
        return None
    return ast.literal_eval(s)


def parse_route(route_str: str) -> Optional[List[int]]:
    val = safe_literal_eval(route_str)
    if val is None:
        return None
    return [int(v) for v in val]


def parse_sorties(sorties_str: str) -> List[Tuple[int, int, int]]:
    val = safe_literal_eval(sorties_str)
    if val is None:
        return []
    return [tuple(int(z) for z in item) for item in val]


def parse_waiting(waiting_str: str) -> Dict[Tuple[int, int], float]:
    val = safe_literal_eval(waiting_str)
    waiting_map = {}
    if val is None:
        return waiting_map
    for item in val:
        arc, w = item
        waiting_map[tuple(int(z) for z in arc)] = float(w)
    return waiting_map


def read_week1_plans_csv(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            parsed = dict(row)
            parsed["n"] = int(row["n"])
            parsed["alpha"] = float(row["alpha"])
            parsed["seed"] = int(row["seed"])
            parsed["route"] = parse_route(row.get("route"))
            parsed["sorties"] = parse_sorties(row.get("sorties"))
            parsed["waiting_map"] = parse_waiting(row.get("waiting"))
            parsed["objective"] = None if row.get("objective") in ("", None, "None") else float(row["objective"])
            parsed["completion_time"] = None if row.get("completion_time") in ("", None, "None") else float(row["completion_time"])
            parsed["runtime"] = None if row.get("runtime") in ("", None, "None") else float(row["runtime"])
            parsed["num_feasible_sorties"] = None if row.get("num_feasible_sorties") in ("", None, "None") else int(float(row["num_feasible_sorties"]))
            rows.append(parsed)
    return rows


# ============================================
# Validation helpers
# ============================================

def route_to_arcs(route: List[int]) -> List[Tuple[int, int]]:
    return [(route[i], route[i + 1]) for i in range(len(route) - 1)]


def validate_week1_plan_against_instance(data: VanRobotData, plan_row: Dict[str, Any]) -> None:
    _, _, A, Fe, start, end = build_sets(data.n)
    route = plan_row["route"]
    sorties = plan_row["sorties"]
    waiting_map = plan_row["waiting_map"]

    if route is None:
        raise ValueError("Week 1 plan has no route")
    if route[0] != start or route[-1] != end:
        raise ValueError(f"Invalid route endpoints: {route[0]}, {route[-1]}")

    route_arcs = route_to_arcs(route)
    for arc in route_arcs:
        if arc not in A:
            raise ValueError(f"Route arc {arc} not in A")

    for sortie in sorties:
        if sortie not in Fe:
            raise ValueError(f"Sortie {sortie} not in Fe")
        k, i, l = sortie
        if (k, l) not in route_arcs:
            raise ValueError(f"Sortie {sortie} not attached to route arc")

    for arc in waiting_map:
        if arc not in A:
            raise ValueError(f"Waiting arc {arc} not in A")


# ============================================
# Week 2 simulation
# ============================================

def replay_deterministic_timing(data: VanRobotData, plan_row: Dict[str, Any]) -> Dict[str, Any]:
    route = plan_row["route"]
    waiting_map = plan_row["waiting_map"]
    _, _, _, _, start, end = build_sets(data.n)

    route_arcs = route_to_arcs(route)
    arrival_times = {start: 0.0}
    for (k, l) in route_arcs:
        arrival_times[l] = arrival_times[k] + data.sV[k] + data.tauV[(k, l)] + waiting_map.get((k, l), 0.0)

    return {
        "route": route,
        "route_arcs": route_arcs,
        "arrival_times": arrival_times,
        "completion_time": arrival_times[end],
    }


def sample_robot_realization(data: VanRobotData, sortie: Tuple[int, int, int], rng: random.Random) -> Dict[str, float]:
    k, i, l = sortie
    tau_out_real = data.tauR_out[(k, i)] + rng.uniform(0.0, data.delta_out[(k, i)])
    service_real = data.sR[i] + rng.uniform(0.0, data.delta_srv[i])
    tau_in_real = data.tauR_in[(i, l)] + rng.uniform(0.0, data.delta_in[(i, l)])
    duration_real = tau_out_real + service_real + tau_in_real
    return {
        "tau_out_real": tau_out_real,
        "service_real": service_real,
        "tau_in_real": tau_in_real,
        "duration_real": duration_real,
    }


def simulate_deterministic_plan_once(data: VanRobotData, plan_row: Dict[str, Any], rng: random.Random) -> Dict[str, Any]:
    timing = replay_deterministic_timing(data, plan_row)
    arrival_times = timing["arrival_times"]
    selected_sorties = plan_row["sorties"]

    violated_sorties = []
    for (k, i, l) in selected_sorties:
        sample = sample_robot_realization(data, (k, i, l), rng)
        realized_robot_completion = arrival_times[k] + data.sV[k] + sample["duration_real"]
        if realized_robot_completion > arrival_times[l] + 1e-9:
            violated_sorties.append((k, i, l))

    return {
        "trial_failed": 1 if violated_sorties else 0,
        "num_sync_violations": len(violated_sorties),
        "violated_sorties": violated_sorties,
        "completion_time_under_uncertainty": timing["completion_time"],
    }


def monte_carlo_deterministic_plan(data: VanRobotData, plan_row: Dict[str, Any], num_trials: int = 1000, sim_seed: int = 100000) -> Dict[str, Any]:
    rng = random.Random(sim_seed)
    failure_count = 0
    total_sync_violations = 0
    max_sync_violations = 0
    completion_times = []

    for _ in range(num_trials):
        res = simulate_deterministic_plan_once(data, plan_row, rng)
        failure_count += res["trial_failed"]
        total_sync_violations += res["num_sync_violations"]
        max_sync_violations = max(max_sync_violations, res["num_sync_violations"])
        completion_times.append(res["completion_time_under_uncertainty"])

    return {
        "num_trials": num_trials,
        "failure_count": failure_count,
        "failure_rate": failure_count / max(1, num_trials),
        "avg_sync_violations": total_sync_violations / max(1, num_trials),
        "max_sync_violations": max_sync_violations,
        "avg_completion_time_under_uncertainty": sum(completion_times) / max(1, len(completion_times)),
    }


# ============================================
# Batch runner
# ============================================

def write_csv(path: str, rows: List[Dict[str, Any]], fieldnames: List[str]) -> None:
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def run_week2_from_existing_week1_plans(
    week1_csv: str,
    num_trials: int = 1000,
    plan_summary_csv: str = "week2_deterministic_plan_summary.csv",
    simulation_csv: str = "week2_deterministic_simulation_results.csv",
):
    week1_rows = read_week1_plans_csv(week1_csv)

    plan_rows = []
    sim_rows = []

    for plan in week1_rows:
        n = plan["n"]
        alpha = plan["alpha"]
        seed = plan["seed"]
        status = str(plan.get("status", "")).strip()

        data = generate_random_euclidean_instance(
            n=n,
            alpha=alpha,
            seed=seed,
            area_size=120.0,
            van_speed=1.0,
            robot_speed=1.3,
            van_cost_per_distance=1.0,
            robot_cost_per_distance=0.4,
            customer_service_time_van=3.0,
            customer_service_time_robot=1.0,
            robot_endurance_factor=1.5,
            gamma=0.10,
            lam=0.01,
            horizon_factor=2.0,
        )

        plan_row = {
            "n": n,
            "alpha": alpha,
            "seed": seed,
            "status": status,
            "objective": plan.get("objective"),
            "route": plan.get("route"),
            "sorties": plan.get("sorties"),
            "waiting": [(arc, round(w, 6)) for arc, w in sorted(plan.get("waiting_map", {}).items())],
            "completion_time": plan.get("completion_time"),
            "runtime": plan.get("runtime"),
            "num_feasible_sorties": plan.get("num_feasible_sorties"),
        }
        plan_rows.append(plan_row)

        if plan.get("route") is None:
            sim_rows.append({
                "n": n,
                "alpha": alpha,
                "seed": seed,
                "num_trials": num_trials,
                "failure_count": None,
                "failure_rate": None,
                "avg_sync_violations": None,
                "max_sync_violations": None,
                "avg_completion_time_under_uncertainty": None,
            })
            continue

        validate_week1_plan_against_instance(data, plan)

        sim = monte_carlo_deterministic_plan(
            data=data,
            plan_row=plan,
            num_trials=num_trials,
            sim_seed=100000 + seed,
        )

        sim_row = {
            "n": n,
            "alpha": alpha,
            "seed": seed,
            "num_trials": sim["num_trials"],
            "failure_count": sim["failure_count"],
            "failure_rate": round(sim["failure_rate"], 6),
            "avg_sync_violations": round(sim["avg_sync_violations"], 6),
            "max_sync_violations": sim["max_sync_violations"],
            "avg_completion_time_under_uncertainty": round(sim["avg_completion_time_under_uncertainty"], 6),
        }
        sim_rows.append(sim_row)

        print(f"\nWeek 2 deterministic stress test | n={n}, alpha={alpha}, seed={seed}")
        print("route:", plan_row["route"])
        print("sorties:", plan_row["sorties"])
        print("waiting:", plan_row["waiting"])
        print("completion time:", plan_row["completion_time"])
        print("runtime:", plan_row["runtime"])
        print("feasible sorties:", plan_row["num_feasible_sorties"])
        print("failure rate:", sim_row["failure_rate"])
        print("avg sync violations:", sim_row["avg_sync_violations"])
        print("avg completion time under uncertainty:", sim_row["avg_completion_time_under_uncertainty"])

    write_csv(
        plan_summary_csv,
        plan_rows,
        ["n", "alpha", "seed", "status", "objective", "route", "sorties", "waiting", "completion_time", "runtime", "num_feasible_sorties"],
    )

    write_csv(
        simulation_csv,
        sim_rows,
        ["n", "alpha", "seed", "num_trials", "failure_count", "failure_rate", "avg_sync_violations", "max_sync_violations", "avg_completion_time_under_uncertainty"],
    )

    print(f"\nSaved Week 2 deterministic plan summary to: {plan_summary_csv}")
    print(f"Saved Week 2 deterministic simulation results to: {simulation_csv}")

    return plan_rows, sim_rows

### Configuration

In [ ]:
WEEK1_RESULTS_CSV = "week1_deterministic_results_ampl.csv"
NUM_TRIALS = 1000
WEEK2_PLAN_SUMMARY_CSV = "week2_deterministic_plan_summary.csv"
WEEK2_SIM_RESULTS_CSV = "week2_deterministic_simulation_results.csv"

plan_rows, sim_rows = run_week2_from_existing_week1_plans(
    week1_csv=WEEK1_RESULTS_CSV,
    num_trials=NUM_TRIALS,
    plan_summary_csv=WEEK2_PLAN_SUMMARY_CSV,
    simulation_csv=WEEK2_SIM_RESULTS_CSV,
)

### Run — Simulation

In [ ]:
for plan in plan_rows:
    n = plan["n"]
    alpha = plan["alpha"]
    seed = plan["seed"]
    data = generate_random_euclidean_instance(n=n, alpha=alpha, seed=seed)
    timing = replay_deterministic_timing(data, {
        "route": plan["route"],
        "waiting_map": dict(plan["waiting"]),
        "sorties": plan["sorties"]
    })
    print(f"\nInstance n={n}, seed={seed}")
    for (k,i,l) in plan["sorties"]:
        nominal = data.tauR_out[(k,i)] + data.sR[i] + data.tauR_in[(i,l)]
        lhs = timing["arrival_times"][l] - (timing["arrival_times"][k] + data.sV[k])
        slack = lhs - nominal
        print((k,i,l), "nominal slack =", round(slack, 8))


---
## Week 3 — Robust Optimization

Extends the MILP with uncertainty sets to produce plans robust to travel-time variability.

In [ ]:


# Cell 2: Week 3 robust AMPL notebook code
# Run this second

from __future__ import annotations

import ast
import csv
import math
import os
import random
import time
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Tuple

from amplpy import AMPL


# ============================================================
# Data container
# ============================================================

@dataclass
class VanRobotData:
    n: int

    # Van parameters
    cV: Dict[Tuple[int, int], float]
    tauV: Dict[Tuple[int, int], float]
    sV: Dict[int, float]

    # Robot nominal parameters
    tauR_out: Dict[Tuple[int, int], float]
    sR: Dict[int, float]
    tauR_in: Dict[Tuple[int, int], float]
    cR: Dict[Tuple[int, int, int], float]

    # Uncertainty parameters
    delta_out: Dict[Tuple[int, int], float]
    delta_srv: Dict[int, float]
    delta_in: Dict[Tuple[int, int], float]

    # Time / policy parameters
    T: Dict[int, float]
    E_R: float
    H: float
    gamma: float
    lam: float

    coords: Optional[Dict[int, Tuple[float, float]]] = None
    alpha: Optional[float] = None


# ============================================================
# Sets
# ============================================================

def build_sets(data: VanRobotData):
    start = 0
    end = data.n + 1
    C = list(range(1, data.n + 1))
    V = [start] + C + [end]
    A = [(k, l) for k in V for l in V if k != l and l != start and k != end]
    Fe = [(k, i, l) for (k, l) in A for i in C if i != k and i != l]
    return V, C, A, Fe, start, end


# ============================================================
# Geometry + instance generator
# SAME parameterization as Week 1 and Week 2
# ============================================================

def euclidean(p: Tuple[float, float], q: Tuple[float, float]) -> float:
    return math.hypot(p[0] - q[0], p[1] - q[1])


def generate_random_euclidean_instance(
    n: int,
    alpha: float,
    seed: int = 0,
    area_size: float = 120.0,
    van_speed: float = 1.0,
    robot_speed: float = 1.3,
    van_cost_per_distance: float = 1.0,
    robot_cost_per_distance: float = 0.4,
    customer_service_time_van: float = 3.0,
    customer_service_time_robot: float = 1.0,
    robot_endurance_factor: float = 1.5,
    gamma: float = 0.10,
    lam: float = 0.01,
    horizon_factor: float = 2.0,
) -> VanRobotData:
    allowed_alphas = {0.05, 0.10, 0.20}
    if alpha not in allowed_alphas:
        raise ValueError(f"alpha must be one of {sorted(allowed_alphas)}, got {alpha}")
    if van_speed <= 0 or robot_speed <= 0:
        raise ValueError("van_speed and robot_speed must be positive")

    rng = random.Random(seed)

    start = 0
    end = n + 1
    C = list(range(1, n + 1))
    V = [start] + C + [end]

    coords = {k: (rng.uniform(0.0, area_size), rng.uniform(0.0, area_size)) for k in V}
    A = [(k, l) for k in V for l in V if k != l and l != start and k != end]

    cV: Dict[Tuple[int, int], float] = {}
    tauV: Dict[Tuple[int, int], float] = {}
    for (k, l) in A:
        d = euclidean(coords[k], coords[l])
        cV[(k, l)] = van_cost_per_distance * d
        tauV[(k, l)] = d / van_speed

    sV = {k: customer_service_time_van for k in V}
    sV[start] = 0.0
    sV[end] = 0.0

    tauR_out: Dict[Tuple[int, int], float] = {}
    tauR_in: Dict[Tuple[int, int], float] = {}
    delta_out: Dict[Tuple[int, int], float] = {}
    delta_in: Dict[Tuple[int, int], float] = {}

    sR = {i: customer_service_time_robot for i in C}
    delta_srv = {i: alpha * sR[i] for i in C}

    for k in V:
        for i in C:
            if i != k:
                d = euclidean(coords[k], coords[i])
                tauR_out[(k, i)] = d / robot_speed
                delta_out[(k, i)] = alpha * tauR_out[(k, i)]

    for i in C:
        for l in V:
            if i != l:
                d = euclidean(coords[i], coords[l])
                tauR_in[(i, l)] = d / robot_speed
                delta_in[(i, l)] = alpha * tauR_in[(i, l)]

    cR: Dict[Tuple[int, int, int], float] = {}
    nominal_sortie_durations: List[float] = []

    for (k, l) in A:
        for i in C:
            if i != k and i != l:
                sortie_dist = euclidean(coords[k], coords[i]) + euclidean(coords[i], coords[l])
                cR[(k, i, l)] = robot_cost_per_distance * sortie_dist
                nominal_sortie_durations.append(
                    tauR_out[(k, i)] + sR[i] + tauR_in[(i, l)]
                )

    avg_nominal_sortie = sum(nominal_sortie_durations) / max(1, len(nominal_sortie_durations))
    E_R = robot_endurance_factor * avg_nominal_sortie

    avg_tauV = sum(tauV.values()) / max(1, len(tauV))
    H = horizon_factor * ((n + 1) * avg_tauV + sum(sV[k] for k in C) + E_R)
    if H <= 0:
        H = 1000.0

    T = {k: H for k in V}

    return VanRobotData(
        n=n,
        cV=cV,
        tauV=tauV,
        sV=sV,
        tauR_out=tauR_out,
        sR=sR,
        tauR_in=tauR_in,
        cR=cR,
        delta_out=delta_out,
        delta_srv=delta_srv,
        delta_in=delta_in,
        T=T,
        E_R=E_R,
        H=H,
        gamma=gamma,
        lam=lam,
        coords=coords,
        alpha=alpha,
    )


# ============================================================
# Validation
# ============================================================

def validate_data_keys(data: VanRobotData, strict: bool = True) -> List[str]:
    V, C, A, Fe, _, _ = build_sets(data)
    missing: List[str] = []

    for k in V:
        if k not in data.sV:
            missing.append(f"sV missing key [{k}]")
        if k not in data.T:
            missing.append(f"T missing key [{k}]")

    for i in C:
        if i not in data.sR:
            missing.append(f"sR missing key [{i}]")
        if i not in data.delta_srv:
            missing.append(f"delta_srv missing key [{i}]")

    for (k, l) in A:
        if (k, l) not in data.cV:
            missing.append(f"cV missing key [{k},{l}]")
        if (k, l) not in data.tauV:
            missing.append(f"tauV missing key [{k},{l}]")

    for (k, i, l) in Fe:
        if (k, i) not in data.tauR_out:
            missing.append(f"tauR_out missing key [{k},{i}]")
        if (i, l) not in data.tauR_in:
            missing.append(f"tauR_in missing key [{i},{l}]")
        if (k, i, l) not in data.cR:
            missing.append(f"cR missing key [{k},{i},{l}]")
        if (k, i) not in data.delta_out:
            missing.append(f"delta_out missing key [{k},{i}]")
        if (i, l) not in data.delta_in:
            missing.append(f"delta_in missing key [{i},{l}]")

    if missing and strict:
        preview = "\n".join(missing[:50])
        more = "" if len(missing) <= 50 else f"\n... and {len(missing)-50} more missing keys"
        raise KeyError(f"Data validation failed with missing keys:\n{preview}{more}")

    return missing


# ============================================================
# Sortie durations and preprocessing
# ============================================================

def compute_nominal_sortie_duration(data: VanRobotData, Fe: Iterable[Tuple[int, int, int]]):
    rho_nom: Dict[Tuple[int, int, int], float] = {}
    for (k, i, l) in Fe:
        rho_nom[(k, i, l)] = (
            data.tauR_out[(k, i)] + data.sR[i] + data.tauR_in[(i, l)]
        )
    return rho_nom


def compute_robust_sortie_duration(data: VanRobotData, Fe: Iterable[Tuple[int, int, int]]):
    rho_rob: Dict[Tuple[int, int, int], float] = {}
    for (k, i, l) in Fe:
        rho_rob[(k, i, l)] = (
            data.tauR_out[(k, i)]
            + data.sR[i]
            + data.tauR_in[(i, l)]
            + data.delta_out[(k, i)]
            + data.delta_srv[i]
            + data.delta_in[(i, l)]
        )
    return rho_rob


def preprocess_sorties_from_duration(
    data: VanRobotData,
    A: Iterable[Tuple[int, int]],
    Fe: Iterable[Tuple[int, int, int]],
    duration: Dict[Tuple[int, int, int], float],
):
    F = [(k, i, l) for (k, i, l) in Fe if duration[(k, i, l)] <= data.E_R]

    W: Dict[Tuple[int, int], float] = {}
    for (k, l) in A:
        vals = [duration[(kk, i, ll)] for (kk, i, ll) in F if kk == k and ll == l]
        W[(k, l)] = max(0.0, max(vals) - data.tauV[(k, l)]) if vals else 0.0

    M: Dict[Tuple[int, int, int], float] = {}
    for (k, i, l) in F:
        M[(k, i, l)] = max(0.0, duration[(k, i, l)] - data.tauV[(k, l)])

    return F, W, M


# ============================================================
# CSV + parsing helpers for Week 1 and Week 2 benchmark files
# ============================================================

def safe_literal_eval(x):
    if x is None:
        return None
    if isinstance(x, (list, tuple, dict, int, float)):
        return x
    s = str(x).strip()
    if s == "" or s.lower() == "none":
        return None
    return ast.literal_eval(s)


def parse_route(route_str: str) -> Optional[List[int]]:
    val = safe_literal_eval(route_str)
    if val is None:
        return None
    return [int(v) for v in val]


def parse_sorties(sorties_str: str) -> List[Tuple[int, int, int]]:
    val = safe_literal_eval(sorties_str)
    if val is None:
        return []
    return [tuple(int(z) for z in item) for item in val]


def parse_waiting(waiting_str: str) -> Dict[Tuple[int, int], float]:
    val = safe_literal_eval(waiting_str)
    waiting_map: Dict[Tuple[int, int], float] = {}
    if val is None:
        return waiting_map
    for item in val:
        arc, w = item
        waiting_map[tuple(int(z) for z in arc)] = float(w)
    return waiting_map


def read_week1_plans_csv(path: str) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with open(path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            parsed = dict(row)
            parsed["n"] = int(row["n"])
            parsed["alpha"] = float(row["alpha"])
            parsed["seed"] = int(row["seed"])
            parsed["route"] = parse_route(row.get("route"))
            parsed["sorties"] = parse_sorties(row.get("sorties"))
            parsed["waiting_map"] = parse_waiting(row.get("waiting"))
            parsed["objective"] = None if row.get("objective") in ("", None, "None") else float(row["objective"])
            parsed["completion_time"] = None if row.get("completion_time") in ("", None, "None") else float(row["completion_time"])
            parsed["runtime"] = None if row.get("runtime") in ("", None, "None") else float(row["runtime"])
            parsed["num_feasible_sorties"] = None if row.get("num_feasible_sorties") in ("", None, "None") else int(float(row["num_feasible_sorties"]))
            rows.append(parsed)
    return rows


def read_week2_simulation_csv(path: str) -> Dict[Tuple[int, float, int], Dict[str, Any]]:
    out: Dict[Tuple[int, float, int], Dict[str, Any]] = {}
    with open(path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            key = (int(row["n"]), float(row["alpha"]), int(row["seed"]))
            out[key] = {
                "n": int(row["n"]),
                "alpha": float(row["alpha"]),
                "seed": int(row["seed"]),
                "num_trials": int(row["num_trials"]) if row.get("num_trials") not in ("", None, "None") else None,
                "failure_count": None if row.get("failure_count") in ("", None, "None") else int(float(row["failure_count"])),
                "failure_rate": None if row.get("failure_rate") in ("", None, "None") else float(row["failure_rate"]),
                "avg_sync_violations": None if row.get("avg_sync_violations") in ("", None, "None") else float(row["avg_sync_violations"]),
                "max_sync_violations": None if row.get("max_sync_violations") in ("", None, "None") else int(float(row["max_sync_violations"])),
                "avg_completion_time_under_uncertainty": None if row.get("avg_completion_time_under_uncertainty") in ("", None, "None") else float(row["avg_completion_time_under_uncertainty"]),
            }
    return out


def write_csv(path: str, rows: List[Dict[str, Any]], fieldnames: List[str]) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


# ============================================================
# Route helpers
# ============================================================

def route_to_arcs(route: List[int]) -> List[Tuple[int, int]]:
    return [(route[idx], route[idx + 1]) for idx in range(len(route) - 1)]


def reconstruct_route(selected_arcs: List[Tuple[int, int]], start: int, end: int) -> List[int]:
    succ: Dict[int, int] = {}
    for (k, l) in selected_arcs:
        if k in succ:
            raise ValueError(f"Multiple successors for node {k}: {succ[k]} and {l}")
        succ[k] = l

    route = [start]
    cur = start
    seen = {start}
    while cur != end:
        if cur not in succ:
            raise ValueError(f"Cannot reconstruct route: no successor for node {cur}")
        cur = succ[cur]
        route.append(cur)
        if cur in seen and cur != end:
            raise ValueError("Cycle detected during route reconstruction")
        seen.add(cur)
        if len(route) > len(selected_arcs) + 2:
            raise ValueError("Route reconstruction exceeded expected length")
    return route


def validate_week1_plan_against_instance(data: VanRobotData, plan_row: Dict[str, Any]) -> None:
    V, C, A, Fe, start, end = build_sets(data)

    route = plan_row["route"]
    sorties = plan_row["sorties"]

    if route is None:
        raise ValueError("Week 1 plan has no route")
    if route[0] != start or route[-1] != end:
        raise ValueError(f"Route endpoints invalid for n={data.n}: {route}")

    route_arcs = route_to_arcs(route)

    for arc in route_arcs:
        if arc not in A:
            raise ValueError(f"Route arc {arc} not in A")

    for sortie in sorties:
        if sortie not in Fe:
            raise ValueError(f"Sortie {sortie} not in Fe")
        k, i, l = sortie
        if (k, l) not in route_arcs:
            raise ValueError(f"Sortie {sortie} is not attached to a route arc")


# ============================================================
# AMPL model writer
# Robust formulation via rho, F, W, M passed from Python
# ============================================================

def write_mod_file(mod_path: str) -> None:
    mod_content = r"""
param n integer > 0;
param start integer := 0;
param end integer := n + 1;

set V;
set A within {V,V};
set F within {V, 1..n, V};

param cV {A} >= 0;
param tauV {A} >= 0;
param sV {V} >= 0;
param cR {F} >= 0;
param rho {F} >= 0;
param W {A} >= 0;
param M {F} >= 0;
param T {V} >= 0;
param H >= 0;
param gamma >= 0;
param lam >= 0;
param bigM_n := n;

var x {A} binary;
var y {F} binary;
var t {V} >= 0;
var w {A} >= 0;
var f {A} >= 0;

minimize total_cost:
    sum {(k,l) in A} cV[k,l] * x[k,l]
  + sum {(k,i,l) in F} cR[k,i,l] * y[k,i,l]
  + gamma * sum {(k,l) in A} w[k,l]
  + lam * t[end];

s.t. serve_once {i in 1..n}:
    sum {(k,i) in A} x[k,i]
  + sum {(k,i,l) in F} y[k,i,l] = 1;

s.t. start_departure_once:
    sum {(start,l) in A} x[start,l] = 1;

s.t. end_arrival_once:
    sum {(k,end) in A} x[k,end] = 1;

s.t. van_flow_balance {i in 1..n}:
    sum {(k,i) in A} x[k,i] = sum {(i,l) in A} x[i,l];

s.t. one_sortie_per_arc {(k,l) in A}:
    sum {(k,i,l) in F} y[k,i,l] <= x[k,l];

s.t. scf_depot_flow:
    sum {(start,l) in A} f[start,l]
  = sum {i in 1..n} sum {(k,i) in A} x[k,i];

s.t. scf_balance {i in 1..n}:
    sum {(k,i) in A} f[k,i] - sum {(i,l) in A} f[i,l]
  = sum {(k,i) in A} x[k,i];

s.t. scf_cap {(k,l) in A}:
    f[k,l] <= bigM_n * x[k,l];

s.t. start_time_zero:
    t[start] = 0;

s.t. time_upper {k in V}:
    t[k] <= T[k];

s.t. time_zero_if_not_visited {i in 1..n}:
    t[i] <= T[i] * sum {(k,i) in A} x[k,i];

s.t. route_horizon:
    t[end] <= H;

s.t. timing {(k,l) in A}:
    x[k,l] ==> (t[l] >= t[k] + sV[k] + tauV[k,l] + w[k,l]);

s.t. sync_wait {(k,i,l) in F}:
    w[k,l] >= rho[k,i,l] - tauV[k,l] - M[k,i,l] * (1 - y[k,i,l]);

s.t. wait_upper {(k,l) in A}:
    w[k,l] <= W[k,l] * x[k,l];
"""
    os.makedirs(os.path.dirname(mod_path) or ".", exist_ok=True)
    with open(mod_path, "w") as f:
        f.write(mod_content)


def write_ampl_data_robust(
    data: VanRobotData,
    dat_path: str,
) -> Dict[str, Any]:
    validate_data_keys(data, strict=True)
    V, C, A, Fe, start, end = build_sets(data)

    rho_rob = compute_robust_sortie_duration(data, Fe)
    F, W, M = preprocess_sorties_from_duration(data, A, Fe, rho_rob)

    os.makedirs(os.path.dirname(dat_path) or ".", exist_ok=True)
    with open(dat_path, "w") as f:
        f.write(f"param n := {data.n};\n\n")
        f.write("set V := " + " ".join(map(str, V)) + ";\n\n")

        f.write("set A :=\n")
        for (k, l) in A:
            f.write(f"  {k} {l}\n")
        f.write(";\n\n")

        f.write("set F :=\n")
        for (k, i, l) in F:
            f.write(f"  {k} {i} {l}\n")
        f.write(";\n\n")

        f.write("param cV :=\n")
        for (k, l) in A:
            f.write(f"  {k} {l} {data.cV[(k,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param tauV :=\n")
        for (k, l) in A:
            f.write(f"  {k} {l} {data.tauV[(k,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param sV :=\n")
        for k in V:
            f.write(f"  {k} {data.sV[k]:.10f}\n")
        f.write(";\n\n")

        f.write("param cR :=\n")
        for (k, i, l) in F:
            f.write(f"  {k} {i} {l} {data.cR[(k,i,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param rho :=\n")
        for (k, i, l) in F:
            f.write(f"  {k} {i} {l} {rho_rob[(k,i,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param W :=\n")
        for (k, l) in A:
            f.write(f"  {k} {l} {W[(k,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param M :=\n")
        for (k, i, l) in F:
            f.write(f"  {k} {i} {l} {M[(k,i,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param T :=\n")
        for k in V:
            f.write(f"  {k} {data.T[k]:.10f}\n")
        f.write(";\n\n")

        f.write(f"param H := {data.H:.10f};\n")
        f.write(f"param gamma := {data.gamma:.10f};\n")
        f.write(f"param lam := {data.lam:.10f};\n")

    return {
        "V": V,
        "C": C,
        "A": A,
        "Fe": Fe,
        "F": F,
        "rho_rob": rho_rob,
        "W": W,
        "M": M,
        "dat_path": dat_path,
    }


# ============================================================
# Solve robust model in AMPL
# ============================================================

def solve_robust_model_ampl(
    data: VanRobotData,
    instance_name: str,
    mod_path: str = "outputs/week3/van_robot_robust.mod",
    dat_dir: str = "outputs/week3/dat",
    time_limit: float = 300.0,
    mip_gap: float = 1e-5,
    log_output: bool = False,
) -> Tuple[Optional[AMPL], Dict[str, Any]]:
    if not os.path.exists(mod_path):
        write_mod_file(mod_path)

    dat_path = os.path.join(dat_dir, f"{instance_name}_robust.dat")
    prep = write_ampl_data_robust(data, dat_path)

    ampl = AMPL()
    ampl.setOption("solver", "gurobi")
    ampl.setOption("solver_msg", 1 if log_output else 0)
    ampl.setOption("gurobi_options", f"timelimit={time_limit} mipgap={mip_gap}")

    ampl.read(mod_path)
    ampl.read_data(dat_path)

    t0 = time.time()
    ampl.solve()
    wall_runtime = time.time() - t0

    status = str(ampl.solve_result)
    if "solved" not in status.lower():
        return None, {
            "status": status,
            "wall_runtime": wall_runtime,
            **prep,
        }

    return ampl, {
        "status": status,
        "wall_runtime": wall_runtime,
        **prep,
    }


# ============================================================
# Robust extraction
# ============================================================

def tuple_key_sort(x):
    if isinstance(x, tuple):
        return tuple(x)
    return x


def ampl_var_dict(ampl: AMPL, var_name: str) -> Dict[Any, float]:
    return ampl.get_variable(var_name).get_values().to_dict()


def compute_objective_breakdown_robust(
    ampl: AMPL,
    data: VanRobotData,
) -> Dict[str, float]:
    x_dict = ampl_var_dict(ampl, "x")
    y_dict = ampl_var_dict(ampl, "y")
    w_dict = ampl_var_dict(ampl, "w")
    t_dict = ampl_var_dict(ampl, "t")
    end = data.n + 1

    van_cost = sum(data.cV.get(a, 0.0) * x_dict.get(a, 0.0) for a in x_dict)
    robot_cost = sum(data.cR.get(s, 0.0) * y_dict.get(s, 0.0) for s in y_dict)
    waiting_cost = data.gamma * sum(w_dict.values())
    completion_cost = data.lam * t_dict[end]
    total = van_cost + robot_cost + waiting_cost + completion_cost

    return {
        "objective_total": total,
        "van_cost": van_cost,
        "robot_cost": robot_cost,
        "waiting_cost": waiting_cost,
        "completion_cost": completion_cost,
        "route_completion_time": t_dict[end],
        "total_waiting_time": sum(w_dict.values()),
    }


def extract_robust_plan(
    ampl: AMPL,
    data: VanRobotData,
    wall_runtime: Optional[float] = None,
) -> Dict[str, Any]:
    x_dict = ampl_var_dict(ampl, "x")
    y_dict = ampl_var_dict(ampl, "y")
    t_dict = ampl_var_dict(ampl, "t")
    w_dict = ampl_var_dict(ampl, "w")

    selected_arcs = sorted([a for a, v in x_dict.items() if v > 0.5], key=tuple_key_sort)
    selected_sorties = sorted([s for s, v in y_dict.items() if v > 0.5], key=tuple_key_sort)
    positive_wait = sorted([(a, w_dict[a]) for a, v in w_dict.items() if v > 1e-8], key=lambda z: tuple_key_sort(z[0]))

    _, _, _, _, start, end = build_sets(data)
    route = reconstruct_route(selected_arcs, start, end)

    bd = compute_objective_breakdown_robust(ampl, data)

    return {
        "status": str(ampl.solve_result),
        "objective": bd["objective_total"],
        "route": route,
        "selected_arcs": selected_arcs,
        "sorties": selected_sorties,
        "waiting": [(arc, round(val, 6)) for arc, val in positive_wait],
        "waiting_map": w_dict,
        "completion_time": bd["route_completion_time"],
        "runtime": wall_runtime,
        "objective_breakdown": bd,
        "arrival_times": t_dict,
    }


# ============================================================
# Week 2 Monte Carlo simulation logic
# Same test for robust plan and deterministic baseline
# ============================================================

def replay_fixed_timing(data: VanRobotData, route: List[int], waiting_map: Dict[Tuple[int, int], float]) -> Dict[str, Any]:
    _, _, _, _, start, end = build_sets(data)
    if route is None:
        raise ValueError("Route is None")
    if route[0] != start or route[-1] != end:
        raise ValueError("Route endpoints do not match depots")

    route_arcs = route_to_arcs(route)
    arrival_times: Dict[int, float] = {start: 0.0}
    for (k, l) in route_arcs:
        arrival_times[l] = arrival_times[k] + data.sV[k] + data.tauV[(k, l)] + waiting_map.get((k, l), 0.0)

    return {
        "route_arcs": route_arcs,
        "arrival_times": arrival_times,
        "completion_time": arrival_times[end],
    }


def sample_robot_realization(
    data: VanRobotData,
    sortie: Tuple[int, int, int],
    rng: random.Random,
) -> Dict[str, float]:
    k, i, l = sortie
    tau_out_real = data.tauR_out[(k, i)] + rng.uniform(0.0, data.delta_out[(k, i)])
    service_real = data.sR[i] + rng.uniform(0.0, data.delta_srv[i])
    tau_in_real = data.tauR_in[(i, l)] + rng.uniform(0.0, data.delta_in[(i, l)])
    duration_real = tau_out_real + service_real + tau_in_real
    return {
        "tau_out_real": tau_out_real,
        "service_real": service_real,
        "tau_in_real": tau_in_real,
        "duration_real": duration_real,
    }


def simulate_fixed_plan_once(
    data: VanRobotData,
    route: List[int],
    sorties: List[Tuple[int, int, int]],
    waiting_map: Dict[Tuple[int, int], float],
    rng: random.Random,
) -> Dict[str, Any]:
    timing = replay_fixed_timing(data, route, waiting_map)
    arrival_times = timing["arrival_times"]

    violated_sorties: List[Tuple[int, int, int]] = []

    for (k, i, l) in sorties:
        sample = sample_robot_realization(data, (k, i, l), rng)
        realized_robot_completion = arrival_times[k] + data.sV[k] + sample["duration_real"]
        if realized_robot_completion > arrival_times[l] + 1e-9:
            violated_sorties.append((k, i, l))

    return {
        "trial_failed": 1 if violated_sorties else 0,
        "num_sync_violations": len(violated_sorties),
        "violated_sorties": violated_sorties,
        "completion_time_under_uncertainty": timing["completion_time"],
    }


def monte_carlo_fixed_plan(
    data: VanRobotData,
    route: List[int],
    sorties: List[Tuple[int, int, int]],
    waiting_map: Dict[Tuple[int, int], float],
    num_trials: int,
    sim_seed: int,
) -> Dict[str, Any]:
    rng = random.Random(sim_seed)

    failure_count = 0
    total_sync_violations = 0
    max_sync_violations = 0
    completion_times: List[float] = []

    for _ in range(num_trials):
        res = simulate_fixed_plan_once(data, route, sorties, waiting_map, rng)
        failure_count += res["trial_failed"]
        total_sync_violations += res["num_sync_violations"]
        max_sync_violations = max(max_sync_violations, res["num_sync_violations"])
        completion_times.append(res["completion_time_under_uncertainty"])

    return {
        "num_trials": num_trials,
        "failure_count": failure_count,
        "failure_rate": failure_count / max(1, num_trials),
        "avg_sync_violations": total_sync_violations / max(1, num_trials),
        "max_sync_violations": max_sync_violations,
        "avg_completion_time_under_uncertainty": sum(completion_times) / max(1, len(completion_times)),
    }


# ============================================================
# Comparison helpers
# ============================================================

def same_route(det_route: Optional[List[int]], rob_route: Optional[List[int]]) -> bool:
    return det_route == rob_route


def same_sorties(det_sorties: List[Tuple[int, int, int]], rob_sorties: List[Tuple[int, int, int]]) -> bool:
    return sorted(det_sorties) == sorted(rob_sorties)


def round_or_none(x, nd=6):
    return None if x is None else round(float(x), nd)


# ============================================================
# Week 3 batch runner
# Reads Week 1 + Week 2 baselines, solves robust AMPL model,
# simulates robust reliability using same Week 2 Monte Carlo test,
# and writes paper-ready CSV outputs.
# ============================================================

def run_week3_robust_comparison(
    week1_csv: str,
    week2_csv: str,
    n_values: List[int],
    alpha_values: List[float],
    seeds: List[int],
    num_trials: int = 1000,
    time_limit: float = 300.0,
    mod_path: str = "outputs/week3/van_robot_robust.mod",
    dat_dir: str = "outputs/week3/dat",
    robust_plan_csv: str = "outputs/week3/week3_robust_plan_summary.csv",
    comparison_csv: str = "outputs/week3/week3_robust_vs_deterministic_comparison.csv",
    robust_sim_csv: str = "outputs/week3/week3_robust_simulation_results.csv",
    log_output: bool = False,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
    week1_rows = read_week1_plans_csv(week1_csv)
    week2_map = read_week2_simulation_csv(week2_csv)

    week1_map: Dict[Tuple[int, float, int], Dict[str, Any]] = {}
    for row in week1_rows:
        week1_map[(row["n"], row["alpha"], row["seed"])] = row

    robust_plan_rows: List[Dict[str, Any]] = []
    robust_sim_rows: List[Dict[str, Any]] = []
    comparison_rows: List[Dict[str, Any]] = []

    for n in n_values:
        for alpha in alpha_values:
            for seed in seeds:
                key = (n, alpha, seed)
                if key not in week1_map:
                    raise ValueError(f"Missing Week 1 baseline row for key={key}")
                if key not in week2_map:
                    raise ValueError(f"Missing Week 2 simulation row for key={key}")

                det_plan = week1_map[key]
                det_sim = week2_map[key]

                data = generate_random_euclidean_instance(
                    n=n,
                    alpha=alpha,
                    seed=seed,
                    area_size=120.0,
                    van_speed=1.0,
                    robot_speed=1.3,
                    van_cost_per_distance=1.0,
                    robot_cost_per_distance=0.4,
                    customer_service_time_van=3.0,
                    customer_service_time_robot=1.0,
                    robot_endurance_factor=1.5,
                    gamma=0.10,
                    lam=0.01,
                    horizon_factor=2.0,
                )

                validate_week1_plan_against_instance(data, det_plan)

                instance_name = f"n{n}_a{int(round(alpha*100)):03d}_s{seed}"
                ampl_obj, solve_info = solve_robust_model_ampl(
                    data=data,
                    instance_name=instance_name,
                    mod_path=mod_path,
                    dat_dir=dat_dir,
                    time_limit=time_limit,
                    log_output=log_output,
                )

                if ampl_obj is None:
                    robust_plan_row = {
                        "n": n,
                        "alpha": alpha,
                        "seed": seed,
                        "status": solve_info["status"],
                        "objective": None,
                        "route": None,
                        "sorties": None,
                        "waiting": None,
                        "completion_time": None,
                        "runtime": round_or_none(solve_info.get("wall_runtime")),
                        "num_feasible_sorties": len(solve_info["F"]),
                    }
                    robust_plan_rows.append(robust_plan_row)

                    robust_sim_row = {
                        "n": n,
                        "alpha": alpha,
                        "seed": seed,
                        "num_trials": num_trials,
                        "failure_count": None,
                        "failure_rate": None,
                        "avg_sync_violations": None,
                        "max_sync_violations": None,
                        "avg_completion_time_under_uncertainty": None,
                    }
                    robust_sim_rows.append(robust_sim_row)

                    comparison_row = {
                        "n": n,
                        "alpha": alpha,
                        "seed": seed,
                        "det_objective": round_or_none(det_plan["objective"]),
                        "rob_objective": None,
                        "objective_gap_rob_minus_det": None,
                        "det_waiting_total": round_or_none(sum(det_plan["waiting_map"].values())),
                        "rob_waiting_total": None,
                        "waiting_gap_rob_minus_det": None,
                        "det_completion_time": round_or_none(det_plan["completion_time"]),
                        "rob_completion_time": None,
                        "completion_gap_rob_minus_det": None,
                        "same_route": None,
                        "same_sorties": None,
                        "det_failure_rate": det_sim["failure_rate"],
                        "rob_failure_rate": None,
                        "reliability_improvement_det_minus_rob_failure": None,
                        "det_avg_sync_violations": det_sim["avg_sync_violations"],
                        "rob_avg_sync_violations": None,
                        "avg_sync_violation_improvement": None,
                    }
                    comparison_rows.append(comparison_row)
                    continue

                rob_plan = extract_robust_plan(ampl_obj, data, wall_runtime=solve_info["wall_runtime"])

                rob_sim = monte_carlo_fixed_plan(
                    data=data,
                    route=rob_plan["route"],
                    sorties=rob_plan["sorties"],
                    waiting_map=rob_plan["waiting_map"],
                    num_trials=num_trials,
                    sim_seed=100000 + seed,
                )

                robust_plan_row = {
                    "n": n,
                    "alpha": alpha,
                    "seed": seed,
                    "status": rob_plan["status"],
                    "objective": round_or_none(rob_plan["objective"]),
                    "route": rob_plan["route"],
                    "sorties": rob_plan["sorties"],
                    "waiting": rob_plan["waiting"],
                    "completion_time": round_or_none(rob_plan["completion_time"]),
                    "runtime": round_or_none(rob_plan["runtime"]),
                    "num_feasible_sorties": len(solve_info["F"]),
                }
                robust_plan_rows.append(robust_plan_row)

                robust_sim_row = {
                    "n": n,
                    "alpha": alpha,
                    "seed": seed,
                    "num_trials": rob_sim["num_trials"],
                    "failure_count": rob_sim["failure_count"],
                    "failure_rate": round_or_none(rob_sim["failure_rate"]),
                    "avg_sync_violations": round_or_none(rob_sim["avg_sync_violations"]),
                    "max_sync_violations": rob_sim["max_sync_violations"],
                    "avg_completion_time_under_uncertainty": round_or_none(rob_sim["avg_completion_time_under_uncertainty"]),
                }
                robust_sim_rows.append(robust_sim_row)

                det_wait_total = sum(det_plan["waiting_map"].values())
                rob_wait_total = sum(rob_plan["waiting_map"].values())

                comparison_row = {
                    "n": n,
                    "alpha": alpha,
                    "seed": seed,
                    "det_objective": round_or_none(det_plan["objective"]),
                    "rob_objective": round_or_none(rob_plan["objective"]),
                    "objective_gap_rob_minus_det": round_or_none(rob_plan["objective"] - det_plan["objective"]),
                    "det_waiting_total": round_or_none(det_wait_total),
                    "rob_waiting_total": round_or_none(rob_wait_total),
                    "waiting_gap_rob_minus_det": round_or_none(rob_wait_total - det_wait_total),
                    "det_completion_time": round_or_none(det_plan["completion_time"]),
                    "rob_completion_time": round_or_none(rob_plan["completion_time"]),
                    "completion_gap_rob_minus_det": round_or_none(rob_plan["completion_time"] - det_plan["completion_time"]),
                    "same_route": same_route(det_plan["route"], rob_plan["route"]),
                    "same_sorties": same_sorties(det_plan["sorties"], rob_plan["sorties"]),
                    "det_failure_rate": round_or_none(det_sim["failure_rate"]),
                    "rob_failure_rate": round_or_none(rob_sim["failure_rate"]),
                    "reliability_improvement_det_minus_rob_failure": round_or_none(det_sim["failure_rate"] - rob_sim["failure_rate"]),
                    "det_avg_sync_violations": round_or_none(det_sim["avg_sync_violations"]),
                    "rob_avg_sync_violations": round_or_none(rob_sim["avg_sync_violations"]),
                    "avg_sync_violation_improvement": round_or_none(det_sim["avg_sync_violations"] - rob_sim["avg_sync_violations"]),
                }
                comparison_rows.append(comparison_row)

                print(f"\nWeek 3 robust comparison | n={n}, alpha={alpha}, seed={seed}")
                print("det route:", det_plan["route"])
                print("rob route:", rob_plan["route"])
                print("det sorties:", det_plan["sorties"])
                print("rob sorties:", rob_plan["sorties"])
                print("det failure rate:", det_sim["failure_rate"])
                print("rob failure rate:", rob_sim["failure_rate"])
                print("objective gap (rob-det):", comparison_row["objective_gap_rob_minus_det"])
                print("waiting gap (rob-det):", comparison_row["waiting_gap_rob_minus_det"])
                print("completion gap (rob-det):", comparison_row["completion_gap_rob_minus_det"])
                print("same route:", comparison_row["same_route"])
                print("same sorties:", comparison_row["same_sorties"])

    write_csv(
        robust_plan_csv,
        robust_plan_rows,
        fieldnames=[
            "n", "alpha", "seed", "status", "objective", "route", "sorties",
            "waiting", "completion_time", "runtime", "num_feasible_sorties"
        ],
    )

    write_csv(
        robust_sim_csv,
        robust_sim_rows,
        fieldnames=[
            "n", "alpha", "seed", "num_trials", "failure_count", "failure_rate",
            "avg_sync_violations", "max_sync_violations", "avg_completion_time_under_uncertainty"
        ],
    )

    write_csv(
        comparison_csv,
        comparison_rows,
        fieldnames=[
            "n", "alpha", "seed",
            "det_objective", "rob_objective", "objective_gap_rob_minus_det",
            "det_waiting_total", "rob_waiting_total", "waiting_gap_rob_minus_det",
            "det_completion_time", "rob_completion_time", "completion_gap_rob_minus_det",
            "same_route", "same_sorties",
            "det_failure_rate", "rob_failure_rate", "reliability_improvement_det_minus_rob_failure",
            "det_avg_sync_violations", "rob_avg_sync_violations", "avg_sync_violation_improvement"
        ],
    )

    print(f"\nSaved robust plan summary to: {robust_plan_csv}")
    print(f"Saved robust simulation results to: {robust_sim_csv}")
    print(f"Saved robust-vs-deterministic comparison to: {comparison_csv}")

    return robust_plan_rows, robust_sim_rows, comparison_rows



### Configuration

In [ ]:
WEEK1_RESULTS_CSV = "week1_deterministic_results_ampl.csv"
WEEK2_RESULTS_CSV = "week2_deterministic_simulation_results.csv"

N_VALUES = [5, 8, 10]
ALPHA_VALUES = [0.10]
SEEDS = [1, 2, 42]

NUM_TRIALS = 1000
TIME_LIMIT = 300.0

ROBUST_PLAN_CSV = "outputs/week3/week3_robust_plan_summary.csv"
ROBUST_SIM_CSV = "outputs/week3/week3_robust_simulation_results.csv"
COMPARISON_CSV = "outputs/week3/week3_robust_vs_deterministic_comparison.csv"

robust_plan_rows, robust_sim_rows, comparison_rows = run_week3_robust_comparison(
    week1_csv=WEEK1_RESULTS_CSV,
    week2_csv=WEEK2_RESULTS_CSV,
    n_values=N_VALUES,
    alpha_values=ALPHA_VALUES,
    seeds=SEEDS,
    num_trials=NUM_TRIALS,
    time_limit=TIME_LIMIT,
    mod_path="outputs/week3/van_robot_robust.mod",
    dat_dir="outputs/week3/dat",
    robust_plan_csv=ROBUST_PLAN_CSV,
    comparison_csv=COMPARISON_CSV,
    robust_sim_csv=ROBUST_SIM_CSV,
    log_output=False,
)

print("\nFirst few robust plan rows:")
for row in robust_plan_rows[:3]:
    print(row)

print("\nFirst few robust simulation rows:")
for row in robust_sim_rows[:3]:
    print(row)

print("\nFirst few comparison rows:")
for row in comparison_rows[:3]:
    print(row)

---
## Week 4 — Comparative Analysis

Full grid comparison of deterministic vs. robust plans across all `(n, α, seed)` combinations.

In [ ]:
from __future__ import annotations
import ast
import csv
import math
import os
import random
import time
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Tuple

from amplpy import AMPL


# ============================================================
# Data container
# ============================================================

@dataclass
class VanRobotData:
    n: int

    cV: Dict[Tuple[int, int], float]
    tauV: Dict[Tuple[int, int], float]
    sV: Dict[int, float]

    tauR_out: Dict[Tuple[int, int], float]
    sR: Dict[int, float]
    tauR_in: Dict[Tuple[int, int], float]
    cR: Dict[Tuple[int, int, int], float]

    delta_out: Dict[Tuple[int, int], float]
    delta_srv: Dict[int, float]
    delta_in: Dict[Tuple[int, int], float]

    T: Dict[int, float]
    E_R: float
    H: float
    gamma: float
    lam: float

    coords: Optional[Dict[int, Tuple[float, float]]] = None
    alpha: Optional[float] = None


# ============================================================
# Sets
# ============================================================

def build_sets(data: VanRobotData):
    start = 0
    end = data.n + 1
    C = list(range(1, data.n + 1))
    V = [start] + C + [end]
    A = [(k, l) for k in V for l in V if k != l and l != start and k != end]
    Fe = [(k, i, l) for (k, l) in A for i in C if i != k and i != l]
    return V, C, A, Fe, start, end


# ============================================================
# Geometry + instance generator
# SAME parameter settings as Weeks 1–3
# ============================================================

def euclidean(p: Tuple[float, float], q: Tuple[float, float]) -> float:
    return math.hypot(p[0] - q[0], p[1] - q[1])


def generate_random_euclidean_instance(
    n: int,
    alpha: float,
    seed: int = 0,
    area_size: float = 120.0,
    van_speed: float = 1.0,
    robot_speed: float = 1.3,
    van_cost_per_distance: float = 1.0,
    robot_cost_per_distance: float = 0.4,
    customer_service_time_van: float = 3.0,
    customer_service_time_robot: float = 1.0,
    robot_endurance_factor: float = 1.5,
    gamma: float = 0.10,
    lam: float = 0.01,
    horizon_factor: float = 2.0,
) -> VanRobotData:
    allowed_alphas = {0.05, 0.10, 0.20}
    if alpha not in allowed_alphas:
        raise ValueError(f"alpha must be one of {sorted(allowed_alphas)}, got {alpha}")
    if van_speed <= 0 or robot_speed <= 0:
        raise ValueError("van_speed and robot_speed must be positive")

    rng = random.Random(seed)

    start = 0
    end = n + 1
    C = list(range(1, n + 1))
    V = [start] + C + [end]

    coords = {k: (rng.uniform(0.0, area_size), rng.uniform(0.0, area_size)) for k in V}
    A = [(k, l) for k in V for l in V if k != l and l != start and k != end]

    cV: Dict[Tuple[int, int], float] = {}
    tauV: Dict[Tuple[int, int], float] = {}
    for (k, l) in A:
        d = euclidean(coords[k], coords[l])
        cV[(k, l)] = van_cost_per_distance * d
        tauV[(k, l)] = d / van_speed

    sV = {k: customer_service_time_van for k in V}
    sV[start] = 0.0
    sV[end] = 0.0

    tauR_out: Dict[Tuple[int, int], float] = {}
    tauR_in: Dict[Tuple[int, int], float] = {}
    delta_out: Dict[Tuple[int, int], float] = {}
    delta_in: Dict[Tuple[int, int], float] = {}

    sR = {i: customer_service_time_robot for i in C}
    delta_srv = {i: alpha * sR[i] for i in C}

    for k in V:
        for i in C:
            if i != k:
                d = euclidean(coords[k], coords[i])
                tauR_out[(k, i)] = d / robot_speed
                delta_out[(k, i)] = alpha * tauR_out[(k, i)]

    for i in C:
        for l in V:
            if i != l:
                d = euclidean(coords[i], coords[l])
                tauR_in[(i, l)] = d / robot_speed
                delta_in[(i, l)] = alpha * tauR_in[(i, l)]

    cR: Dict[Tuple[int, int, int], float] = {}
    nominal_sortie_durations: List[float] = []

    for (k, l) in A:
        for i in C:
            if i != k and i != l:
                sortie_dist = euclidean(coords[k], coords[i]) + euclidean(coords[i], coords[l])
                cR[(k, i, l)] = robot_cost_per_distance * sortie_dist
                nominal_sortie_durations.append(
                    tauR_out[(k, i)] + sR[i] + tauR_in[(i, l)]
                )

    avg_nominal_sortie = sum(nominal_sortie_durations) / max(1, len(nominal_sortie_durations))
    E_R = robot_endurance_factor * avg_nominal_sortie

    avg_tauV = sum(tauV.values()) / max(1, len(tauV))
    H = horizon_factor * ((n + 1) * avg_tauV + sum(sV[k] for k in C) + E_R)
    if H <= 0:
        H = 1000.0

    T = {k: H for k in V}

    return VanRobotData(
        n=n,
        cV=cV,
        tauV=tauV,
        sV=sV,
        tauR_out=tauR_out,
        sR=sR,
        tauR_in=tauR_in,
        cR=cR,
        delta_out=delta_out,
        delta_srv=delta_srv,
        delta_in=delta_in,
        T=T,
        E_R=E_R,
        H=H,
        gamma=gamma,
        lam=lam,
        coords=coords,
        alpha=alpha,
    )


# ============================================================
# Validation
# ============================================================

def validate_data_keys(data: VanRobotData, strict: bool = True) -> List[str]:
    V, C, A, Fe, _, _ = build_sets(data)
    missing: List[str] = []

    for k in V:
        if k not in data.sV:
            missing.append(f"sV missing key [{k}]")
        if k not in data.T:
            missing.append(f"T missing key [{k}]")

    for i in C:
        if i not in data.sR:
            missing.append(f"sR missing key [{i}]")
        if i not in data.delta_srv:
            missing.append(f"delta_srv missing key [{i}]")

    for (k, l) in A:
        if (k, l) not in data.cV:
            missing.append(f"cV missing key [{k},{l}]")
        if (k, l) not in data.tauV:
            missing.append(f"tauV missing key [{k},{l}]")

    for (k, i, l) in Fe:
        if (k, i) not in data.tauR_out:
            missing.append(f"tauR_out missing key [{k},{i}]")
        if (i, l) not in data.tauR_in:
            missing.append(f"tauR_in missing key [{i},{l}]")
        if (k, i, l) not in data.cR:
            missing.append(f"cR missing key [{k},{i},{l}]")
        if (k, i) not in data.delta_out:
            missing.append(f"delta_out missing key [{k},{i}]")
        if (i, l) not in data.delta_in:
            missing.append(f"delta_in missing key [{i},{l}]")

    if missing and strict:
        preview = "\n".join(missing[:50])
        more = "" if len(missing) <= 50 else f"\n... and {len(missing)-50} more missing keys"
        raise KeyError(f"Data validation failed with missing keys:\n{preview}{more}")

    return missing


# ============================================================
# Sortie durations and preprocessing
# ============================================================

def compute_nominal_sortie_duration(data: VanRobotData, Fe: Iterable[Tuple[int, int, int]]):
    rho_nom: Dict[Tuple[int, int, int], float] = {}
    for (k, i, l) in Fe:
        rho_nom[(k, i, l)] = (
            data.tauR_out[(k, i)] + data.sR[i] + data.tauR_in[(i, l)]
        )
    return rho_nom


def compute_robust_sortie_duration(data: VanRobotData, Fe: Iterable[Tuple[int, int, int]]):
    rho_rob: Dict[Tuple[int, int, int], float] = {}
    for (k, i, l) in Fe:
        rho_rob[(k, i, l)] = (
            data.tauR_out[(k, i)]
            + data.sR[i]
            + data.tauR_in[(i, l)]
            + data.delta_out[(k, i)]
            + data.delta_srv[i]
            + data.delta_in[(i, l)]
        )
    return rho_rob


def preprocess_sorties_from_duration(
    data: VanRobotData,
    A: Iterable[Tuple[int, int]],
    Fe: Iterable[Tuple[int, int, int]],
    duration: Dict[Tuple[int, int, int], float],
):
    F = [(k, i, l) for (k, i, l) in Fe if duration[(k, i, l)] <= data.E_R]

    W: Dict[Tuple[int, int], float] = {}
    for (k, l) in A:
        vals = [duration[(kk, i, ll)] for (kk, i, ll) in F if kk == k and ll == l]
        W[(k, l)] = max(0.0, max(vals) - data.tauV[(k, l)]) if vals else 0.0

    M: Dict[Tuple[int, int, int], float] = {}
    for (k, i, l) in F:
        M[(k, i, l)] = max(0.0, duration[(k, i, l)] - data.tauV[(k, l)])

    return F, W, M


# ============================================================
# Route helpers
# ============================================================

def route_to_arcs(route: List[int]) -> List[Tuple[int, int]]:
    return [(route[idx], route[idx + 1]) for idx in range(len(route) - 1)]


def reconstruct_route(selected_arcs: List[Tuple[int, int]], start: int, end: int) -> List[int]:
    succ: Dict[int, int] = {}
    for (k, l) in selected_arcs:
        if k in succ:
            raise ValueError(f"Multiple successors for node {k}: {succ[k]} and {l}")
        succ[k] = l

    route = [start]
    cur = start
    seen = {start}
    while cur != end:
        if cur not in succ:
            raise ValueError(f"Cannot reconstruct route: no successor for node {cur}")
        cur = succ[cur]
        route.append(cur)
        if cur in seen and cur != end:
            raise ValueError("Cycle detected during route reconstruction")
        seen.add(cur)
        if len(route) > len(selected_arcs) + 2:
            raise ValueError("Route reconstruction exceeded expected length")
    return route


# ============================================================
# AMPL model writer
# One model, data controls deterministic vs robust through rho/F/W/M
# ============================================================

def write_mod_file(mod_path: str) -> None:
    mod_content = r"""
param n integer > 0;
param start integer := 0;
param end integer := n + 1;

set V;
set A within {V,V};
set F within {V, 1..n, V};

param cV {A} >= 0;
param tauV {A} >= 0;
param sV {V} >= 0;
param cR {F} >= 0;
param rho {F} >= 0;
param W {A} >= 0;
param M {F} >= 0;
param T {V} >= 0;
param H >= 0;
param gamma >= 0;
param lam >= 0;
param bigM_n := n;

var x {A} binary;
var y {F} binary;
var t {V} >= 0;
var w {A} >= 0;
var f {A} >= 0;

minimize total_cost:
    sum {(k,l) in A} cV[k,l] * x[k,l]
  + sum {(k,i,l) in F} cR[k,i,l] * y[k,i,l]
  + gamma * sum {(k,l) in A} w[k,l]
  + lam * t[end];

s.t. serve_once {i in 1..n}:
    sum {(k,i) in A} x[k,i]
  + sum {(k,i,l) in F} y[k,i,l] = 1;

s.t. start_departure_once:
    sum {(start,l) in A} x[start,l] = 1;

s.t. end_arrival_once:
    sum {(k,end) in A} x[k,end] = 1;

s.t. van_flow_balance {i in 1..n}:
    sum {(k,i) in A} x[k,i] = sum {(i,l) in A} x[i,l];

s.t. one_sortie_per_arc {(k,l) in A}:
    sum {(k,i,l) in F} y[k,i,l] <= x[k,l];

s.t. scf_depot_flow:
    sum {(start,l) in A} f[start,l]
  = sum {i in 1..n} sum {(k,i) in A} x[k,i];

s.t. scf_balance {i in 1..n}:
    sum {(k,i) in A} f[k,i] - sum {(i,l) in A} f[i,l]
  = sum {(k,i) in A} x[k,i];

s.t. scf_cap {(k,l) in A}:
    f[k,l] <= bigM_n * x[k,l];

s.t. start_time_zero:
    t[start] = 0;

s.t. time_upper {k in V}:
    t[k] <= T[k];

s.t. time_zero_if_not_visited {i in 1..n}:
    t[i] <= T[i] * sum {(k,i) in A} x[k,i];

s.t. route_horizon:
    t[end] <= H;

s.t. timing {(k,l) in A}:
    x[k,l] ==> (t[l] >= t[k] + sV[k] + tauV[k,l] + w[k,l]);

s.t. sync_wait {(k,i,l) in F}:
    w[k,l] >= rho[k,i,l] - tauV[k,l] - M[k,i,l] * (1 - y[k,i,l]);

s.t. wait_upper {(k,l) in A}:
    w[k,l] <= W[k,l] * x[k,l];
"""
    os.makedirs(os.path.dirname(mod_path) or ".", exist_ok=True)
    with open(mod_path, "w") as f:
        f.write(mod_content)


def write_ampl_data_for_mode(
    data: VanRobotData,
    mode: str,
    dat_path: str,
) -> Dict[str, Any]:
    validate_data_keys(data, strict=True)
    V, C, A, Fe, start, end = build_sets(data)

    if mode == "robust":
        sortie_duration = compute_robust_sortie_duration(data, Fe)
    elif mode == "deterministic":
        sortie_duration = compute_nominal_sortie_duration(data, Fe)
    else:
        raise ValueError("mode must be 'deterministic' or 'robust'")

    F, W, M = preprocess_sorties_from_duration(data, A, Fe, sortie_duration)

    os.makedirs(os.path.dirname(dat_path) or ".", exist_ok=True)
    with open(dat_path, "w") as f:
        f.write(f"param n := {data.n};\n\n")
        f.write("set V := " + " ".join(map(str, V)) + ";\n\n")

        f.write("set A :=\n")
        for (k, l) in A:
            f.write(f"  {k} {l}\n")
        f.write(";\n\n")

        f.write("set F :=\n")
        for (k, i, l) in F:
            f.write(f"  {k} {i} {l}\n")
        f.write(";\n\n")

        f.write("param cV :=\n")
        for (k, l) in A:
            f.write(f"  {k} {l} {data.cV[(k,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param tauV :=\n")
        for (k, l) in A:
            f.write(f"  {k} {l} {data.tauV[(k,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param sV :=\n")
        for k in V:
            f.write(f"  {k} {data.sV[k]:.10f}\n")
        f.write(";\n\n")

        f.write("param cR :=\n")
        for (k, i, l) in F:
            f.write(f"  {k} {i} {l} {data.cR[(k,i,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param rho :=\n")
        for (k, i, l) in F:
            f.write(f"  {k} {i} {l} {sortie_duration[(k,i,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param W :=\n")
        for (k, l) in A:
            f.write(f"  {k} {l} {W[(k,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param M :=\n")
        for (k, i, l) in F:
            f.write(f"  {k} {i} {l} {M[(k,i,l)]:.10f}\n")
        f.write(";\n\n")

        f.write("param T :=\n")
        for k in V:
            f.write(f"  {k} {data.T[k]:.10f}\n")
        f.write(";\n\n")

        f.write(f"param H := {data.H:.10f};\n")
        f.write(f"param gamma := {data.gamma:.10f};\n")
        f.write(f"param lam := {data.lam:.10f};\n")

    return {
        "mode": mode,
        "V": V,
        "C": C,
        "A": A,
        "Fe": Fe,
        "F": F,
        "sortie_duration": sortie_duration,
        "W": W,
        "M": M,
        "dat_path": dat_path,
    }


# ============================================================
# Solve via AMPL + Gurobi
# ============================================================

def solve_model_ampl(
    data: VanRobotData,
    mode: str,
    instance_name: str,
    mod_path: str,
    dat_dir: str,
    time_limit: float = 300.0,
    mip_gap: float = 1e-5,
    log_output: bool = False,
) -> Tuple[Optional[AMPL], Dict[str, Any]]:
    if not os.path.exists(mod_path):
        write_mod_file(mod_path)

    dat_path = os.path.join(dat_dir, f"{instance_name}_{mode}.dat")
    prep = write_ampl_data_for_mode(data, mode, dat_path)

    ampl_obj = AMPL()
    ampl_obj.setOption("solver", "gurobi")
    ampl_obj.setOption("solver_msg", 1 if log_output else 0)
    ampl_obj.setOption("gurobi_options", f"timelimit={time_limit} mipgap={mip_gap}")

    ampl_obj.read(mod_path)
    ampl_obj.read_data(dat_path)

    t0 = time.time()
    ampl_obj.solve()
    wall_runtime = time.time() - t0

    status = str(ampl_obj.solve_result)
    if "solved" not in status.lower():
        return None, {
            "status": status,
            "wall_runtime": wall_runtime,
            **prep,
        }

    return ampl_obj, {
        "status": status,
        "wall_runtime": wall_runtime,
        **prep,
    }


# ============================================================
# Solution extraction
# ============================================================

def ampl_var_dict(ampl_obj: AMPL, var_name: str) -> Dict[Any, float]:
    return ampl_obj.get_variable(var_name).get_values().to_dict()


def tuple_key_sort(x):
    return tuple(x) if isinstance(x, tuple) else x


def compute_objective_breakdown(
    ampl_obj: AMPL,
    data: VanRobotData,
) -> Dict[str, float]:
    x_dict = ampl_var_dict(ampl_obj, "x")
    y_dict = ampl_var_dict(ampl_obj, "y")
    w_dict = ampl_var_dict(ampl_obj, "w")
    t_dict = ampl_var_dict(ampl_obj, "t")
    end = data.n + 1

    van_cost = sum(data.cV.get(a, 0.0) * x_dict.get(a, 0.0) for a in x_dict)
    robot_cost = sum(data.cR.get(s, 0.0) * y_dict.get(s, 0.0) for s in y_dict)
    waiting_cost = data.gamma * sum(w_dict.values())
    completion_cost = data.lam * t_dict[end]
    total = van_cost + robot_cost + waiting_cost + completion_cost

    return {
        "objective_total": total,
        "van_cost": van_cost,
        "robot_cost": robot_cost,
        "waiting_cost": waiting_cost,
        "completion_cost": completion_cost,
        "route_completion_time": t_dict[end],
        "total_waiting_time": sum(w_dict.values()),
    }


def extract_plan_from_ampl(
    ampl_obj: AMPL,
    data: VanRobotData,
    mode: str,
    wall_runtime: Optional[float] = None,
) -> Dict[str, Any]:
    x_dict = ampl_var_dict(ampl_obj, "x")
    y_dict = ampl_var_dict(ampl_obj, "y")
    t_dict = ampl_var_dict(ampl_obj, "t")
    w_dict = ampl_var_dict(ampl_obj, "w")

    selected_arcs = sorted([a for a, v in x_dict.items() if v > 0.5], key=tuple_key_sort)
    selected_sorties = sorted([s for s, v in y_dict.items() if v > 0.5], key=tuple_key_sort)
    positive_wait = sorted([(a, w_dict[a]) for a, v in w_dict.items() if v > 1e-8], key=lambda z: tuple_key_sort(z[0]))

    _, _, _, _, start, end = build_sets(data)
    route = reconstruct_route(selected_arcs, start, end)

    bd = compute_objective_breakdown(ampl_obj, data)

    return {
        "mode": mode,
        "status": str(ampl_obj.solve_result),
        "objective": bd["objective_total"],
        "route": route,
        "selected_arcs": selected_arcs,
        "sorties": selected_sorties,
        "waiting": [(arc, round(val, 6)) for arc, val in positive_wait],
        "waiting_map": w_dict,
        "completion_time": bd["route_completion_time"],
        "runtime": wall_runtime,
        "objective_breakdown": bd,
        "arrival_times": t_dict,
    }


# ============================================================
# Week 2 / Week 4 Monte Carlo simulation logic
# Deterministic and robust plans are both tested using the same fixed-policy simulation
# ============================================================

def replay_fixed_timing(
    data: VanRobotData,
    route: List[int],
    waiting_map: Dict[Tuple[int, int], float],
) -> Dict[str, Any]:
    _, _, _, _, start, end = build_sets(data)

    if route is None:
        raise ValueError("Route is None")
    if route[0] != start or route[-1] != end:
        raise ValueError("Route endpoints do not match depots")

    route_arcs = route_to_arcs(route)
    arrival_times: Dict[int, float] = {start: 0.0}
    for (k, l) in route_arcs:
        arrival_times[l] = arrival_times[k] + data.sV[k] + data.tauV[(k, l)] + waiting_map.get((k, l), 0.0)

    return {
        "route_arcs": route_arcs,
        "arrival_times": arrival_times,
        "completion_time": arrival_times[end],
    }


def sample_robot_realization(
    data: VanRobotData,
    sortie: Tuple[int, int, int],
    rng: random.Random,
) -> Dict[str, float]:
    k, i, l = sortie
    tau_out_real = data.tauR_out[(k, i)] + rng.uniform(0.0, data.delta_out[(k, i)])
    service_real = data.sR[i] + rng.uniform(0.0, data.delta_srv[i])
    tau_in_real = data.tauR_in[(i, l)] + rng.uniform(0.0, data.delta_in[(i, l)])
    duration_real = tau_out_real + service_real + tau_in_real
    return {
        "tau_out_real": tau_out_real,
        "service_real": service_real,
        "tau_in_real": tau_in_real,
        "duration_real": duration_real,
    }


def simulate_fixed_plan_once(
    data: VanRobotData,
    route: List[int],
    sorties: List[Tuple[int, int, int]],
    waiting_map: Dict[Tuple[int, int], float],
    rng: random.Random,
) -> Dict[str, Any]:
    timing = replay_fixed_timing(data, route, waiting_map)
    arrival_times = timing["arrival_times"]

    violated_sorties: List[Tuple[int, int, int]] = []

    for (k, i, l) in sorties:
        sample = sample_robot_realization(data, (k, i, l), rng)
        realized_robot_completion = arrival_times[k] + data.sV[k] + sample["duration_real"]
        if realized_robot_completion > arrival_times[l] + 1e-9:
            violated_sorties.append((k, i, l))

    return {
        "trial_failed": 1 if violated_sorties else 0,
        "num_sync_violations": len(violated_sorties),
        "violated_sorties": violated_sorties,
        "completion_time_under_uncertainty": timing["completion_time"],
    }


def monte_carlo_fixed_plan(
    data: VanRobotData,
    route: List[int],
    sorties: List[Tuple[int, int, int]],
    waiting_map: Dict[Tuple[int, int], float],
    num_trials: int,
    sim_seed: int,
) -> Dict[str, Any]:
    rng = random.Random(sim_seed)

    failure_count = 0
    total_sync_violations = 0
    max_sync_violations = 0
    completion_times: List[float] = []

    for _ in range(num_trials):
        res = simulate_fixed_plan_once(data, route, sorties, waiting_map, rng)
        failure_count += res["trial_failed"]
        total_sync_violations += res["num_sync_violations"]
        max_sync_violations = max(max_sync_violations, res["num_sync_violations"])
        completion_times.append(res["completion_time_under_uncertainty"])

    return {
        "num_trials": num_trials,
        "failure_count": failure_count,
        "failure_rate": failure_count / max(1, num_trials),
        "avg_sync_violations": total_sync_violations / max(1, num_trials),
        "max_sync_violations": max_sync_violations,
        "avg_completion_time_under_uncertainty": sum(completion_times) / max(1, len(completion_times)),
    }


# ============================================================
# CSV parsing helpers for reusing existing files if desired
# ============================================================

def safe_literal_eval(x):
    if x is None:
        return None
    if isinstance(x, (list, tuple, dict, int, float)):
        return x
    s = str(x).strip()
    if s == "" or s.lower() == "none":
        return None
    return ast.literal_eval(s)


def parse_route(route_str: str) -> Optional[List[int]]:
    val = safe_literal_eval(route_str)
    if val is None:
        return None
    return [int(v) for v in val]


def parse_sorties(sorties_str: str) -> List[Tuple[int, int, int]]:
    val = safe_literal_eval(sorties_str)
    if val is None:
        return []
    return [tuple(int(z) for z in item) for item in val]


def parse_waiting(waiting_str: str) -> Dict[Tuple[int, int], float]:
    val = safe_literal_eval(waiting_str)
    waiting_map: Dict[Tuple[int, int], float] = {}
    if val is None:
        return waiting_map
    for item in val:
        arc, w = item
        waiting_map[tuple(int(z) for z in arc)] = float(w)
    return waiting_map


def read_plan_csv(path: str) -> Dict[Tuple[int, float, int], Dict[str, Any]]:
    out: Dict[Tuple[int, float, int], Dict[str, Any]] = {}
    if not os.path.exists(path):
        return out

    with open(path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            key = (int(row["n"]), float(row["alpha"]), int(row["seed"]))
            out[key] = {
                "n": int(row["n"]),
                "alpha": float(row["alpha"]),
                "seed": int(row["seed"]),
                "status": row.get("status"),
                "objective": None if row.get("objective") in ("", None, "None") else float(row["objective"]),
                "route": parse_route(row.get("route")),
                "sorties": parse_sorties(row.get("sorties")),
                "waiting": safe_literal_eval(row.get("waiting")),
                "waiting_map": parse_waiting(row.get("waiting")),
                "completion_time": None if row.get("completion_time") in ("", None, "None") else float(row["completion_time"]),
                "runtime": None if row.get("runtime") in ("", None, "None") else float(row["runtime"]),
                "num_feasible_sorties": None if row.get("num_feasible_sorties") in ("", None, "None") else int(float(row["num_feasible_sorties"])),
            }
    return out


def read_sim_csv(path: str) -> Dict[Tuple[int, float, int], Dict[str, Any]]:
    out: Dict[Tuple[int, float, int], Dict[str, Any]] = {}
    if not os.path.exists(path):
        return out

    with open(path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            key = (int(row["n"]), float(row["alpha"]), int(row["seed"]))
            out[key] = {
                "n": int(row["n"]),
                "alpha": float(row["alpha"]),
                "seed": int(row["seed"]),
                "num_trials": None if row.get("num_trials") in ("", None, "None") else int(float(row["num_trials"])),
                "failure_count": None if row.get("failure_count") in ("", None, "None") else int(float(row["failure_count"])),
                "failure_rate": None if row.get("failure_rate") in ("", None, "None") else float(row["failure_rate"]),
                "avg_sync_violations": None if row.get("avg_sync_violations") in ("", None, "None") else float(row["avg_sync_violations"]),
                "max_sync_violations": None if row.get("max_sync_violations") in ("", None, "None") else int(float(row["max_sync_violations"])),
                "avg_completion_time_under_uncertainty": None if row.get("avg_completion_time_under_uncertainty") in ("", None, "None") else float(row["avg_completion_time_under_uncertainty"]),
            }
    return out


# ============================================================
# Row builders
# ============================================================

def round_or_none(x, nd=6):
    return None if x is None else round(float(x), nd)


def plan_row_from_plan(
    n: int,
    alpha: float,
    seed: int,
    plan: Dict[str, Any],
) -> Dict[str, Any]:
    return {
        "n": n,
        "alpha": alpha,
        "seed": seed,
        "status": plan["status"],
        "objective": round_or_none(plan["objective"]),
        "route": plan["route"],
        "sorties": plan["sorties"],
        "waiting": plan["waiting"],
        "completion_time": round_or_none(plan["completion_time"]),
        "runtime": round_or_none(plan["runtime"]),
        "num_feasible_sorties": plan.get("num_feasible_sorties"),
    }


def sim_row_from_sim(
    n: int,
    alpha: float,
    seed: int,
    sim: Dict[str, Any],
) -> Dict[str, Any]:
    return {
        "n": n,
        "alpha": alpha,
        "seed": seed,
        "num_trials": sim["num_trials"],
        "failure_count": sim["failure_count"],
        "failure_rate": round_or_none(sim["failure_rate"]),
        "avg_sync_violations": round_or_none(sim["avg_sync_violations"]),
        "max_sync_violations": sim["max_sync_violations"],
        "avg_completion_time_under_uncertainty": round_or_none(sim["avg_completion_time_under_uncertainty"]),
    }


# ============================================================
# Week 1 deterministic baseline on full alpha grid
# ============================================================

def run_week1_all_alpha_ampl(
    n_values: List[int],
    alpha_values: List[float],
    seeds: List[int],
    time_limit: float,
    mod_path: str,
    dat_dir: str,
    output_csv: str,
    log_output: bool = False,
) -> Dict[Tuple[int, float, int], Dict[str, Any]]:
    results: Dict[Tuple[int, float, int], Dict[str, Any]] = {}
    rows: List[Dict[str, Any]] = []

    for n in n_values:
        for alpha in alpha_values:
            for seed in seeds:
                data = generate_random_euclidean_instance(n=n, alpha=alpha, seed=seed)
                instance_name = f"n{n}_a{int(round(alpha*100)):03d}_s{seed}"

                ampl_obj, solve_info = solve_model_ampl(
                    data=data,
                    mode="deterministic",
                    instance_name=instance_name,
                    mod_path=mod_path,
                    dat_dir=dat_dir,
                    time_limit=time_limit,
                    log_output=log_output,
                )

                if ampl_obj is None:
                    plan = {
                        "status": solve_info["status"],
                        "objective": None,
                        "route": None,
                        "sorties": [],
                        "waiting": [],
                        "waiting_map": {},
                        "completion_time": None,
                        "runtime": solve_info["wall_runtime"],
                        "num_feasible_sorties": len(solve_info["F"]),
                    }
                else:
                    plan = extract_plan_from_ampl(
                        ampl_obj=ampl_obj,
                        data=data,
                        mode="deterministic",
                        wall_runtime=solve_info["wall_runtime"],
                    )
                    plan["num_feasible_sorties"] = len(solve_info["F"])

                key = (n, alpha, seed)
                results[key] = plan
                rows.append(plan_row_from_plan(n, alpha, seed, plan))

                print(f"Week 1 deterministic | n={n}, alpha={alpha}, seed={seed}")
                print("status:", plan["status"])
                print("route:", plan["route"])
                print("sorties:", plan["sorties"])
                print("waiting:", plan["waiting"])
                print("completion time:", plan["completion_time"])
                print("runtime:", plan["runtime"])
                print("feasible sorties:", plan["num_feasible_sorties"])
                print()

    write_csv(
        output_csv,
        rows,
        fieldnames=[
            "n", "alpha", "seed", "status", "objective", "route", "sorties",
            "waiting", "completion_time", "runtime", "num_feasible_sorties"
        ],
    )
    return results


# ============================================================
# Week 2 deterministic simulation on full alpha grid
# ============================================================

def run_week2_all_alpha_from_week1(
    week1_results: Dict[Tuple[int, float, int], Dict[str, Any]],
    n_values: List[int],
    alpha_values: List[float],
    seeds: List[int],
    num_trials: int,
    output_csv: str,
) -> Dict[Tuple[int, float, int], Dict[str, Any]]:
    sim_results: Dict[Tuple[int, float, int], Dict[str, Any]] = {}
    rows: List[Dict[str, Any]] = []

    for n in n_values:
        for alpha in alpha_values:
            for seed in seeds:
                key = (n, alpha, seed)
                if key not in week1_results:
                    raise ValueError(f"Missing Week 1 plan for key={key}")

                plan = week1_results[key]
                data = generate_random_euclidean_instance(n=n, alpha=alpha, seed=seed)

                if plan["route"] is None:
                    sim = {
                        "num_trials": num_trials,
                        "failure_count": None,
                        "failure_rate": None,
                        "avg_sync_violations": None,
                        "max_sync_violations": None,
                        "avg_completion_time_under_uncertainty": None,
                    }
                else:
                    sim = monte_carlo_fixed_plan(
                        data=data,
                        route=plan["route"],
                        sorties=plan["sorties"],
                        waiting_map=plan["waiting_map"],
                        num_trials=num_trials,
                        sim_seed=100000 + seed,
                    )

                sim_results[key] = sim
                rows.append(sim_row_from_sim(n, alpha, seed, sim))

                print(f"Week 2 deterministic simulation | n={n}, alpha={alpha}, seed={seed}")
                print("failure rate:", sim["failure_rate"])
                print("avg sync violations:", sim["avg_sync_violations"])
                print("avg completion time:", sim["avg_completion_time_under_uncertainty"])
                print()

    write_csv(
        output_csv,
        rows,
        fieldnames=[
            "n", "alpha", "seed", "num_trials", "failure_count", "failure_rate",
            "avg_sync_violations", "max_sync_violations", "avg_completion_time_under_uncertainty"
        ],
    )
    return sim_results


# ============================================================
# Week 3/4 robust comparison on full alpha grid
# ============================================================

def same_route(det_route: Optional[List[int]], rob_route: Optional[List[int]]) -> bool:
    return det_route == rob_route


def same_sorties(det_sorties: List[Tuple[int, int, int]], rob_sorties: List[Tuple[int, int, int]]) -> bool:
    return sorted(det_sorties) == sorted(rob_sorties)


def run_week4_full_grid(
    week1_results: Dict[Tuple[int, float, int], Dict[str, Any]],
    week2_results: Dict[Tuple[int, float, int], Dict[str, Any]],
    n_values: List[int],
    alpha_values: List[float],
    seeds: List[int],
    num_trials: int,
    time_limit: float,
    mod_path: str,
    dat_dir: str,
    robust_plan_csv: str,
    robust_sim_csv: str,
    comparison_csv: str,
    alpha_summary_csv: str,
    log_output: bool = False,
) -> Tuple[
    Dict[Tuple[int, float, int], Dict[str, Any]],
    Dict[Tuple[int, float, int], Dict[str, Any]],
    List[Dict[str, Any]],
    List[Dict[str, Any]],
    List[Dict[str, Any]],
    List[Dict[str, Any]],
]:
    robust_plans: Dict[Tuple[int, float, int], Dict[str, Any]] = {}
    robust_sims: Dict[Tuple[int, float, int], Dict[str, Any]] = {}

    robust_plan_rows: List[Dict[str, Any]] = []
    robust_sim_rows: List[Dict[str, Any]] = []
    comparison_rows: List[Dict[str, Any]] = []

    for n in n_values:
        for alpha in alpha_values:
            for seed in seeds:
                key = (n, alpha, seed)

                if key not in week1_results:
                    raise ValueError(f"Missing Week 1 deterministic result for key={key}")
                if key not in week2_results:
                    raise ValueError(f"Missing Week 2 deterministic simulation result for key={key}")

                det_plan = week1_results[key]
                det_sim = week2_results[key]
                data = generate_random_euclidean_instance(n=n, alpha=alpha, seed=seed)
                instance_name = f"n{n}_a{int(round(alpha*100)):03d}_s{seed}"

                ampl_obj, solve_info = solve_model_ampl(
                    data=data,
                    mode="robust",
                    instance_name=instance_name,
                    mod_path=mod_path,
                    dat_dir=dat_dir,
                    time_limit=time_limit,
                    log_output=log_output,
                )

                if ampl_obj is None:
                    rob_plan = {
                        "status": solve_info["status"],
                        "objective": None,
                        "route": None,
                        "sorties": [],
                        "waiting": [],
                        "waiting_map": {},
                        "completion_time": None,
                        "runtime": solve_info["wall_runtime"],
                        "num_feasible_sorties": len(solve_info["F"]),
                    }
                    rob_sim = {
                        "num_trials": num_trials,
                        "failure_count": None,
                        "failure_rate": None,
                        "avg_sync_violations": None,
                        "max_sync_violations": None,
                        "avg_completion_time_under_uncertainty": None,
                    }
                else:
                    rob_plan = extract_plan_from_ampl(
                        ampl_obj=ampl_obj,
                        data=data,
                        mode="robust",
                        wall_runtime=solve_info["wall_runtime"],
                    )
                    rob_plan["num_feasible_sorties"] = len(solve_info["F"])

                    rob_sim = monte_carlo_fixed_plan(
                        data=data,
                        route=rob_plan["route"],
                        sorties=rob_plan["sorties"],
                        waiting_map=rob_plan["waiting_map"],
                        num_trials=num_trials,
                        sim_seed=100000 + seed,
                    )

                robust_plans[key] = rob_plan
                robust_sims[key] = rob_sim

                robust_plan_rows.append(plan_row_from_plan(n, alpha, seed, rob_plan))
                robust_sim_rows.append(sim_row_from_sim(n, alpha, seed, rob_sim))

                det_wait_total = None if det_plan["route"] is None else sum(det_plan["waiting_map"].values())
                rob_wait_total = None if rob_plan["route"] is None else sum(rob_plan["waiting_map"].values())

                comparison_row = {
                    "n": n,
                    "alpha": alpha,
                    "seed": seed,

                    "det_objective": round_or_none(det_plan["objective"]),
                    "rob_objective": round_or_none(rob_plan["objective"]),
                    "objective_gap_rob_minus_det": None if det_plan["objective"] is None or rob_plan["objective"] is None else round_or_none(rob_plan["objective"] - det_plan["objective"]),

                    "det_waiting_total": round_or_none(det_wait_total),
                    "rob_waiting_total": round_or_none(rob_wait_total),
                    "waiting_gap_rob_minus_det": None if det_wait_total is None or rob_wait_total is None else round_or_none(rob_wait_total - det_wait_total),

                    "det_completion_time": round_or_none(det_plan["completion_time"]),
                    "rob_completion_time": round_or_none(rob_plan["completion_time"]),
                    "completion_gap_rob_minus_det": None if det_plan["completion_time"] is None or rob_plan["completion_time"] is None else round_or_none(rob_plan["completion_time"] -
det_plan["completion_time"]),

                    "same_route": None if det_plan["route"] is None or rob_plan["route"] is None else same_route(det_plan["route"], rob_plan["route"]),
                    "same_sorties": None if det_plan["route"] is None or rob_plan["route"] is None else same_sorties(det_plan["sorties"], rob_plan["sorties"]),

                    "det_failure_rate": round_or_none(det_sim["failure_rate"]),
                    "rob_failure_rate": round_or_none(rob_sim["failure_rate"]),
                    "reliability_improvement_det_minus_rob_failure": None if det_sim["failure_rate"] is None or rob_sim["failure_rate"] is None else round_or_none(det_sim["failure_rate"] -
rob_sim["failure_rate"]),

                    "det_avg_sync_violations": round_or_none(det_sim["avg_sync_violations"]),
                    "rob_avg_sync_violations": round_or_none(rob_sim["avg_sync_violations"]),
                    "avg_sync_violation_improvement": None if det_sim["avg_sync_violations"] is None or rob_sim["avg_sync_violations"] is None else round_or_none(det_sim["avg_sync_violations"] -
rob_sim["avg_sync_violations"]),
                }
                comparison_rows.append(comparison_row)

                print(f"Week 4 robust comparison | n={n}, alpha={alpha}, seed={seed}")
                print("det route:", det_plan["route"])
                print("rob route:", rob_plan["route"])
                print("det sorties:", det_plan["sorties"])
                print("rob sorties:", rob_plan["sorties"])
                print("det failure rate:", det_sim["failure_rate"])
                print("rob failure rate:", rob_sim["failure_rate"])
                print("objective gap (rob-det):", comparison_row["objective_gap_rob_minus_det"])
                print("waiting gap (rob-det):", comparison_row["waiting_gap_rob_minus_det"])
                print("completion gap (rob-det):", comparison_row["completion_gap_rob_minus_det"])
                print("same route:", comparison_row["same_route"])
                print("same sorties:", comparison_row["same_sorties"])
                print()

    write_csv(
        robust_plan_csv,
        robust_plan_rows,
        fieldnames=[
            "n", "alpha", "seed", "status", "objective", "route", "sorties",
            "waiting", "completion_time", "runtime", "num_feasible_sorties"
        ],
    )

    write_csv(
        robust_sim_csv,
        robust_sim_rows,
        fieldnames=[
            "n", "alpha", "seed", "num_trials", "failure_count", "failure_rate",
            "avg_sync_violations", "max_sync_violations", "avg_completion_time_under_uncertainty"
        ],
    )

    write_csv(
        comparison_csv,
        comparison_rows,
        fieldnames=[
            "n", "alpha", "seed",
            "det_objective", "rob_objective", "objective_gap_rob_minus_det",
            "det_waiting_total", "rob_waiting_total", "waiting_gap_rob_minus_det",
            "det_completion_time", "rob_completion_time", "completion_gap_rob_minus_det",
            "same_route", "same_sorties",
            "det_failure_rate", "rob_failure_rate", "reliability_improvement_det_minus_rob_failure",
            "det_avg_sync_violations", "rob_avg_sync_violations", "avg_sync_violation_improvement"
        ],
    )

    # alpha-level summary
    alpha_summary_rows: List[Dict[str, Any]] = []
    for alpha in alpha_values:
        rows_alpha = [r for r in comparison_rows if r["alpha"] == alpha]

        det_failure_vals = [r["det_failure_rate"] for r in rows_alpha if r["det_failure_rate"] is not None]
        rob_failure_vals = [r["rob_failure_rate"] for r in rows_alpha if r["rob_failure_rate"] is not None]
        obj_gap_vals = [r["objective_gap_rob_minus_det"] for r in rows_alpha if r["objective_gap_rob_minus_det"] is not None]
        wait_gap_vals = [r["waiting_gap_rob_minus_det"] for r in rows_alpha if r["waiting_gap_rob_minus_det"] is not None]
        comp_gap_vals = [r["completion_gap_rob_minus_det"] for r in rows_alpha if r["completion_gap_rob_minus_det"] is not None]
        same_route_vals = [r["same_route"] for r in rows_alpha if r["same_route"] is not None]
        same_sorties_vals = [r["same_sorties"] for r in rows_alpha if r["same_sorties"] is not None]

        alpha_summary_rows.append({
            "alpha": alpha,
            "num_instances": len(rows_alpha),
            "avg_deterministic_failure_rate": round_or_none(sum(det_failure_vals) / len(det_failure_vals)) if det_failure_vals else None,
            "avg_robust_failure_rate": round_or_none(sum(rob_failure_vals) / len(rob_failure_vals)) if rob_failure_vals else None,
            "avg_objective_gap_rob_minus_det": round_or_none(sum(obj_gap_vals) / len(obj_gap_vals)) if obj_gap_vals else None,
            "avg_waiting_gap_rob_minus_det": round_or_none(sum(wait_gap_vals) / len(wait_gap_vals)) if wait_gap_vals else None,
            "avg_completion_gap_rob_minus_det": round_or_none(sum(comp_gap_vals) / len(comp_gap_vals)) if comp_gap_vals else None,
            "route_same_count": sum(1 for v in same_route_vals if v),
            "route_same_fraction": round_or_none(sum(1 for v in same_route_vals if v) / len(same_route_vals)) if same_route_vals else None,
            "sorties_same_count": sum(1 for v in same_sorties_vals if v),
            "sorties_same_fraction": round_or_none(sum(1 for v in same_sorties_vals if v) / len(same_sorties_vals)) if same_sorties_vals else None,
        })

    write_csv(
        alpha_summary_csv,
        alpha_summary_rows,
        fieldnames=[
            "alpha", "num_instances",
            "avg_deterministic_failure_rate", "avg_robust_failure_rate",
            "avg_objective_gap_rob_minus_det", "avg_waiting_gap_rob_minus_det", "avg_completion_gap_rob_minus_det",
            "route_same_count", "route_same_fraction",
            "sorties_same_count", "sorties_same_fraction"
        ],
    )

    return (
        robust_plans,
        robust_sims,
        robust_plan_rows,
        robust_sim_rows,
        comparison_rows,
        alpha_summary_rows,
    )

### Configuration

In [ ]:
ALPHA_VALUES = [0.05, 0.10, 0.20]
N_VALUES = [5, 8, 10]
SEEDS = [1, 2, 42]

NUM_TRIALS = 1000
TIME_LIMIT = 300.0

WEEK1_FULL_CSV = "week1_deterministic_results_full_grid.csv"

WEEK2_FULL_CSV = "week2_deterministic_simulation_results_full_grid.csv"

ROBUST_PLAN_CSV = "week4_robust_plan_summary.csv"
ROBUST_SIM_CSV = "week4_robust_simulation_results.csv"
COMPARISON_CSV = "week4_robust_vs_deterministic_comparison.csv"
ALPHA_SUMMARY_CSV = "week4_alpha_summary.csv"

(
    robust_plans,
    robust_sims,
    robust_plan_rows,
    robust_sim_rows,
    comparison_rows,
    alpha_summary_rows,
) = run_week4_full_grid(
    week1_results=week1_results,
    week2_results=week2_results,
    n_values=N_VALUES,
    alpha_values=ALPHA_VALUES,
    seeds=SEEDS,
    num_trials=NUM_TRIALS,
    time_limit=TIME_LIMIT,
    mod_path=MOD_PATH,
    dat_dir=DAT_DIR,
    robust_plan_csv=ROBUST_PLAN_CSV,
    robust_sim_csv=ROBUST_SIM_CSV,
    comparison_csv=COMPARISON_CSV,
    alpha_summary_csv=ALPHA_SUMMARY_CSV,
    log_output=False,
)

print("Saved:")
print(ROBUST_PLAN_CSV)
print(ROBUST_SIM_CSV)
print(COMPARISON_CSV)
print(ALPHA_SUMMARY_CSV)

### Run — Full Grid

In [ ]:
# Cell 5: Run Week 1 deterministic baseline for all alpha values

WEEK1_FULL_CSV = "week1_deterministic_results_full_grid.csv"
MOD_PATH = "outputs/week4/van_robot_week4.mod"
DAT_DIR = "outputs/week4/dat"

week1_results = run_week1_all_alpha_ampl(
    n_values=N_VALUES,
    alpha_values=ALPHA_VALUES,
    seeds=SEEDS,
    time_limit=TIME_LIMIT,
    mod_path=MOD_PATH,
    dat_dir=DAT_DIR,
    output_csv=WEEK1_FULL_CSV,
    log_output=False,
)

print("Saved:", WEEK1_FULL_CSV)

In [ ]:
# Cell 6: Run Week 2 deterministic simulation for all alpha values

WEEK2_FULL_CSV = "week2_deterministic_simulation_results_full_grid.csv"

week2_results = run_week2_all_alpha_from_week1(
    week1_results=week1_results,
    n_values=N_VALUES,
    alpha_values=ALPHA_VALUES,
    seeds=SEEDS,
    num_trials=NUM_TRIALS,
    output_csv=WEEK2_FULL_CSV,
)

print("Saved:", WEEK2_FULL_CSV)

### Analysis — Plan Comparison

In [ ]:
import pandas as pd

robust_plan_df = pd.read_csv("week4_robust_plan_summary.csv")
robust_sim_df = pd.read_csv("week4_robust_simulation_results.csv")
comparison_df = pd.read_csv("week4_robust_vs_deterministic_comparison.csv")
alpha_summary_df = pd.read_csv("week4_alpha_summary.csv")

print("Robust plan summary:")
display(robust_plan_df.head())

print("Robust simulation summary:")
display(robust_sim_df.head())

print("Robust vs deterministic comparison:")
display(comparison_df.head())

print("Alpha-level summary:")
display(alpha_summary_df)

In [ ]:
# Find the alpha = 0.20 cases where route or sorties changed

changed_cases = [
    row for row in comparison_rows
    if row["alpha"] == 0.20 and (
        row["same_route"] is False or row["same_sorties"] is False
    )
]

print("Number of changed-structure cases at alpha=0.20:", len(changed_cases))
print()

for row in changed_cases:
    print("n =", row["n"], "| alpha =", row["alpha"], "| seed =", row["seed"])
    print("det_objective:", row["det_objective"])
    print("rob_objective:", row["rob_objective"])
    print("objective_gap_rob_minus_det:", row["objective_gap_rob_minus_det"])
    print("det_waiting_total:", row["det_waiting_total"])
    print("rob_waiting_total:", row["rob_waiting_total"])
    print("waiting_gap_rob_minus_det:", row["waiting_gap_rob_minus_det"])
    print("det_completion_time:", row["det_completion_time"])
    print("rob_completion_time:", row["rob_completion_time"])
    print("completion_gap_rob_minus_det:", row["completion_gap_rob_minus_det"])
    print("same_route:", row["same_route"])
    print("same_sorties:", row["same_sorties"])
    print("det_failure_rate:", row["det_failure_rate"])
    print("rob_failure_rate:", row["rob_failure_rate"])
    print("det_avg_sync_violations:", row["det_avg_sync_violations"])
    print("rob_avg_sync_violations:", row["rob_avg_sync_violations"])
    print("-" * 80)

In [ ]:
# Print the actual deterministic and robust plans for changed cases at alpha=0.20

for row in changed_cases:
    key = (row["n"], row["alpha"], row["seed"])
    det_plan = week1_results[key]
    rob_plan = robust_plans[key]

    print("\n============================================================")
    print(f"Changed case -> n={row['n']}, alpha={row['alpha']}, seed={row['seed']}")
    print("============================================================")

    print("\nDeterministic route:")
    print(det_plan["route"])
    print("Deterministic sorties:")
    print(det_plan["sorties"])
    print("Deterministic waiting:")
    print(det_plan["waiting"])
    print("Deterministic completion time:", det_plan["completion_time"])
    print("Deterministic objective:", det_plan["objective"])

    print("\nRobust route:")
    print(rob_plan["route"])
    print("Robust sorties:")
    print(rob_plan["sorties"])
    print("Robust waiting:")
    print(rob_plan["waiting"])
    print("Robust completion time:", rob_plan["completion_time"])
    print("Robust objective:", rob_plan["objective"])

In [ ]:
import pandas as pd

changed_df = pd.DataFrame(changed_cases)
changed_df

In [ ]:
changed_df[[
    "n", "alpha", "seed", "same_route", "same_sorties",
    "objective_gap_rob_minus_det",
    "waiting_gap_rob_minus_det",
    "completion_gap_rob_minus_det"
]]

### Export — Changed Structure Cases

In [ ]:
import csv

changed_case_rows = [
    row for row in comparison_rows
    if row["alpha"] == 0.20 and (
        row["same_route"] is False or row["same_sorties"] is False
    )
]

with open("week4_changed_structure_cases.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(changed_case_rows[0].keys()))
    writer.writeheader()
    writer.writerows(changed_case_rows)

print("Saved: week4_changed_structure_cases.csv")

In [ ]:
interesting_rows = []

for key in [(5, 0.20, 1), (5, 0.20, 42)]:
    det_plan = week1_results[key]
    rob_plan = robust_plans[key]

    interesting_rows.append({
        "n": key[0],
        "alpha": key[1],
        "seed": key[2],
        "det_route": det_plan["route"],
        "det_sorties": det_plan["sorties"],
        "det_waiting": det_plan["waiting"],
        "det_completion_time": det_plan["completion_time"],
        "det_objective": det_plan["objective"],
        "rob_route": rob_plan["route"],
        "rob_sorties": rob_plan["sorties"],
        "rob_waiting": rob_plan["waiting"],
        "rob_completion_time": rob_plan["completion_time"],
        "rob_objective": rob_plan["objective"],
    })

with open("week4_changed_structure_plan_details.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(interesting_rows[0].keys()))
    writer.writeheader()
    writer.writerows(interesting_rows)

print("Saved: week4_changed_structure_plan_details.csv")